In [1]:
import json
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
LS_FILE = Path("mplads_india_ls18.json")
RS_FILE = Path("mplads_india_rs.json")


def load_json_records(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # Our current files are expected to contain a list of records.
    if isinstance(data, list):
        return data

    # Safety fallback in case the JSON is wrapped in a dictionary.
    if isinstance(data, dict):
        for key in ["data", "records", "results", "works"]:
            if key in data and isinstance(data[key], list):
                return data[key]

    raise ValueError(f"Unsupported JSON structure in {path}")


ls_records = load_json_records(LS_FILE)
rs_records = load_json_records(RS_FILE)

print("Lok Sabha records:", len(ls_records))
print("Rajya Sabha records:", len(rs_records))

Lok Sabha records: 18498
Rajya Sabha records: 4607


In [3]:
def normalize_record(record, house, source_file):
    mp_details = record.get("mp_details") or {}

    raw_district = record.get("district")

    location = record.get("location")

    # In these datasets, location commonly starts with the geographic
    # district name followed by administrative information in parentheses.
    geo_district = None

    if isinstance(location, str) and location.strip():
        geo_district = location.split("(", 1)[0].strip()

    return {
        "source_id": record.get("_id"),
        "source_file": source_file,

        "work_id": record.get("work_id"),
        "house": house,

        "work_description": record.get("work_description"),
        "category": record.get("category"),

        "mp_name": mp_details.get("name"),
        "constituency": mp_details.get("constituency"),

        "state": record.get("state"),

        # Preserve original value
        "source_district": raw_district,

        # Derived geographic value
        "geo_district": geo_district,

        "location": location,

        "cost": record.get("cost"),

        "completion_date": record.get("completion_date"),
        "completion_year": record.get("completion_year"),

        "beneficiaries": record.get("beneficiaries")
    }

In [4]:
ls_normalized = [
    normalize_record(
        record,
        house="Lok Sabha",
        source_file="mplads_india_ls18.json"
    )
    for record in ls_records
]

rs_normalized = [
    normalize_record(
        record,
        house="Rajya Sabha",
        source_file="mplads_india_rs.json"
    )
    for record in rs_records
]

In [5]:
ls_df = pd.DataFrame(ls_normalized)
rs_df = pd.DataFrame(rs_normalized)

In [6]:
df = pd.concat(
    [ls_df, rs_df],
    ignore_index=True
)

In [7]:
print(df.shape)
print(df["house"].value_counts())

(23105, 16)
house
Lok Sabha      18498
Rajya Sabha     4607
Name: count, dtype: int64


In [8]:
df["cost"] = pd.to_numeric(
    df["cost"],
    errors="coerce"
)

In [9]:
df["completion_date"] = pd.to_datetime(
    df["completion_date"],
    errors="coerce",
    utc=True
)

In [10]:
def clean_text(value):
    if pd.isna(value):
        return None

    value = str(value)

    # Normalize whitespace
    value = " ".join(value.split())

    # Lowercase
    value = value.lower()

    return value.strip()

In [11]:
df["work_description_clean"] = (
    df["work_description"]
    .apply(clean_text)
)

In [12]:
df["state"] = (
    df["state"]
    .astype("string")
    .str.strip()
    .str.upper()
)

In [13]:
df["category"] = (
    df["category"]
    .astype("string")
    .str.strip()
    .str.upper()
)

In [14]:
missing_work_id = df["work_id"].isna().sum()
print("Missing work IDs:", missing_work_id)

Missing work IDs: 0


In [15]:
duplicate_work_ids = df["work_id"].duplicated(keep=False)
print(
    "Duplicate work ID records:",
    duplicate_work_ids.sum()
)

Duplicate work ID records: 0


In [16]:
missing_cost = df["cost"].isna().sum()
print("Missing cost:", missing_cost)

Missing cost: 0


In [17]:
invalid_cost = (df["cost"] <= 0).sum()
print("Non-positive costs:", invalid_cost)

Non-positive costs: 0


In [18]:
df["cost_missing_flag"] = df["cost"].isna()
df["cost_invalid_flag"] = (
    df["cost"].notna() &
    (df["cost"] <= 0)
)

In [19]:
df["description_missing_flag"] = (
    df["work_description_clean"].isna() |
    (df["work_description_clean"] == "")
)

In [20]:
print(
    "Missing descriptions:",
    df["description_missing_flag"].sum()
)

Missing descriptions: 0


In [21]:
df["date_missing_or_invalid_flag"] = (
    df["completion_date"].isna()
)

In [22]:
print(
    "Invalid/missing dates:",
    df["date_missing_or_invalid_flag"].sum()
)

Invalid/missing dates: 0


In [23]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23105 entries, 0 to 23104
Data columns (total 21 columns):
 #   Column                        Non-Null Count  Dtype              
---  ------                        --------------  -----              
 0   source_id                     23105 non-null  object             
 1   source_file                   23105 non-null  object             
 2   work_id                       23105 non-null  int64              
 3   house                         23105 non-null  object             
 4   work_description              23105 non-null  object             
 5   category                      23105 non-null  string             
 6   mp_name                       23105 non-null  object             
 7   constituency                  23105 non-null  object             
 8   state                         23105 non-null  string             
 9   source_district               23105 non-null  object             
 10  geo_district                  2310

In [24]:
print(
    df.groupby("house")["source_district"]
      .value_counts()
      .head(20)
)

house      source_district
Lok Sabha  MACHHLISHAHR       448
           PURI               308
           KAUSHAMBI          306
           NIZAMABAD          296
           BHONGIR            284
           BHADOHI            249
           FIROZPUR           208
           BETUL              179
           JAUNPUR            178
           NAGINA             170
           KANCHEEPURAM       167
           FARIDKOT           164
           KHERI              152
           DIAMOND HARBOUR    149
           NAVSARI            144
           KHEDA              142
           UDHAMPUR           142
           FARRUKHABAD        139
           MAHABUBABAD        137
           ANAND              134
Name: count, dtype: int64


In [25]:
print(
    df.groupby("house")["geo_district"]
      .nunique()
)

house
Lok Sabha      635
Rajya Sabha    352
Name: geo_district, dtype: int64


In [26]:
quality_report = pd.DataFrame({
    "metric": [
        "total_records",
        "unique_work_ids",
        "duplicate_work_id_records",
        "missing_work_ids",
        "missing_cost",
        "invalid_cost",
        "missing_description",
        "missing_or_invalid_dates",
        "unique_states",
        "unique_geo_districts"
    ],

    "value": [
        len(df),
        df["work_id"].nunique(dropna=True),
        df["work_id"].duplicated(keep=False).sum(),
        df["work_id"].isna().sum(),
        df["cost"].isna().sum(),
        (df["cost"] <= 0).sum(),
        df["description_missing_flag"].sum(),
        df["date_missing_or_invalid_flag"].sum(),
        df["state"].nunique(),
        df["geo_district"].nunique()
    ]
})

In [27]:
print(quality_report)

                      metric  value
0              total_records  23105
1            unique_work_ids  23105
2  duplicate_work_id_records      0
3           missing_work_ids      0
4               missing_cost      0
5               invalid_cost      0
6        missing_description      0
7   missing_or_invalid_dates      0
8              unique_states     33
9       unique_geo_districts    670


In [28]:
quality_report.to_csv(
    "data_quality_report.csv",
    index=False
)

In [29]:
OUTPUT_DIR = Path("artha_pipeline")
OUTPUT_DIR.mkdir(exist_ok=True)

df.to_csv(
    OUTPUT_DIR / "combined_real_normalized.csv",
    index=False
)

quality_report.to_csv(
    OUTPUT_DIR / "data_quality_report.csv",
    index=False
)

#Step 2 : Work-Type Classification

In [30]:
# ============================================================
# ARTHASENTINEL
# STEP 2 — WORK TYPE + WORK SUBTYPE CLASSIFICATION
# ============================================================
#
# INPUT:
#   df from Step 1
#
# IMPORTANT INPUT COLUMN:
#   work_description_clean
#
# OUTPUT COLUMNS:
#   primary_work_type
#   secondary_work_type
#   work_subtype
#   work_type_confidence
#   work_type_match_count
#   work_type_scores
#
# OUTPUT FILE:
#   artha_pipeline/combined_real_with_work_type.csv
#
# ============================================================


import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 0. VERIFY INPUT
# ============================================================

if "df" not in globals():
    raise RuntimeError(
        "DataFrame 'df' was not found. "
        "Run Step 1 first."
    )

required_columns = [
    "work_description_clean"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

print("=" * 75)
print("STEP 2 — WORK TYPE CLASSIFICATION")
print("=" * 75)

print("\nInput records:", len(df))


# ============================================================
# 1. BROAD WORK-TYPE TAXONOMY
# ============================================================

WORK_TYPES = [
    "ROAD",
    "WATER",
    "BUILDING",
    "DRAINAGE",
    "ELECTRIFICATION",
    "SANITATION",
    "OTHER"
]


# ============================================================
# 2. TEXT NORMALIZATION
# ============================================================
#
# Our actual dataset contains spelling variations such as:
#
#   pum          → pump
#   draiange     → drainage
#   installtion  → installation
#
# We normalize common domain-specific variations before
# classification.
#
# ============================================================

TEXT_REPLACEMENTS = {

    # --------------------------------------------------------
    # Common spelling errors
    # --------------------------------------------------------

    "draiange": "drainage",
    "draiage": "drainage",
    "draiange": "drainage",

    "pum set": "pump set",
    "pumset": "pump set",
    "pumpset": "pump set",
    "pum": "pump",

    "installtion": "installation",
    "instalation": "installation",
    "instllation": "installation",

    "augumenatation": "augmentation",
    "augumentation": "augmentation",

    "repiars": "repairs",
    "repairs": "repairs",

    # --------------------------------------------------------
    # Lighting variations
    # --------------------------------------------------------

    "high mass light": "high mast light",
    "high mass lights": "high mast lights",
    "highmast": "high mast",

    "street lights": "street light",
    "streetlights": "street light",

    "led lights": "led light",

    # --------------------------------------------------------
    # Water terminology
    # --------------------------------------------------------

    "roplant": "ro plant",
    "roplant ": "ro plant",

    "handpumps": "hand pump",

    "bore wells": "borewell",
    "bore well": "borewell",

    "pipe line": "pipeline",

    # --------------------------------------------------------
    # Road terminology
    # --------------------------------------------------------

    "c.c. road": "cc road",
    "c.c road": "cc road",
    "c c road": "cc road",

    "p.c.c road": "pcc road",
    "p.c.c. road": "pcc road",

    "paver blocks": "paver block",

    "interlok tiles": "interlock tiles",

    # --------------------------------------------------------
    # Miscellaneous
    # --------------------------------------------------------

    "toilet block": "toilet block",
    "commuity hall": "community hall",
    "community hll": "community hall"
}


def normalize_description(text):
    """
    Apply lightweight domain-specific normalization.

    We intentionally do NOT use aggressive spell correction because
    project names, village names and technical terms should be
    preserved.
    """

    if pd.isna(text):
        return ""

    text = str(text).lower().strip()

    # Normalize whitespace
    text = " ".join(text.split())

    # Apply replacements
    for old, new in TEXT_REPLACEMENTS.items():
        text = text.replace(old, new)

    return text


# Create normalized text specifically for classification
df["work_description_model"] = (
    df["work_description_clean"]
    .apply(normalize_description)
)


# ============================================================
# 3. BROAD TYPE RULES
# ============================================================
#
# Each phrase has a weight.
#
# Highly specific phrase:
#       high weight
#
# Generic word:
#       low weight
#
# This prevents:
#
# "High Mast Light near Main Road"
#
# from being classified as ROAD simply because "road" appears.
#
# ============================================================

WORK_TYPE_RULES = {

    # ========================================================
    # ROAD
    # ========================================================

    "ROAD": {

        "cc road": 6,
        "pcc road": 6,
        "concrete road": 6,
        "cement road": 6,

        "paver road": 6,
        "paver block road": 6,
        "interlock road": 6,
        "interlocking road": 6,

        "bt road": 6,
        "black top road": 6,

        "road construction": 5,
        "road development": 5,
        "road improvement": 5,
        "road work": 4,

        "paver block": 5,
        "paver tiles": 5,
        "interlock tiles": 5,

        "pavement": 4,
        "pathway": 3,

        # Generic terms get LOW weight
        "road": 1,
        "lane": 1,
        "street": 1
    },


    # ========================================================
    # WATER
    # ========================================================

    "WATER": {

        "drinking water": 7,
        "water supply": 7,
        "water harvesting": 7,

        "mineral water plant": 8,
        "water plant": 7,

        "ro system": 7,
        "ro plant": 7,

        "borewell": 7,
        "hand pump": 7,
        "handpump": 7,

        "submersible pump": 8,
        "pump set": 6,
        "motor pump": 6,
        "motor and pump": 6,

        "water tank": 6,
        "water tanker": 6,

        "water pipeline": 7,
        "pipeline": 5,

        "water source": 5,

        # generic
        "water": 1
    },


    # ========================================================
    # DRAINAGE
    # ========================================================

    "DRAINAGE": {

        "storm water drain": 8,
        "storm drain": 8,

        "underground drainage": 8,
        "under ground drainage": 8,

        "drainage": 7,
        "drain construction": 7,

        "side drainage": 7,
        "side drain": 7,
        "box drain": 7,

        "earthen drain": 6,

        "culvert": 5,

        "nali": 5,
        "nala": 5,

        "drain": 2
    },


    # ========================================================
    # ELECTRIFICATION
    # ========================================================

    "ELECTRIFICATION": {

        "high mast light": 9,
        "high mast lights": 9,

        "semi high mast": 9,
        "semi-high mast": 9,

        "solar street light": 9,
        "solar street lights": 9,

        "street light": 8,

        "led street light": 9,
        "led street lights": 9,

        "led light": 7,

        "solar light": 7,
        "solar lights": 7,

        "lighting arrangement": 7,

        "lamp post": 7,
        "light pole": 7,

        "electrical work": 7,
        "electrification": 8,

        "electrical line": 6,
        "electric line": 6,

        "power line": 6,
        "lt line": 6,
        "ht line": 6,
        "11 kv line": 7,

        "transformer": 6,

        "solar power plant": 7,

        # Generic
        "lighting": 2,
        "light": 1,
        "lamp": 1,
        "solar": 2
    },


    # ========================================================
    # SANITATION
    # ========================================================

    "SANITATION": {

        "toilet block": 9,
        "public toilet": 9,
        "community toilet": 9,
        "school toilet": 9,

        "toilet": 8,

        "sanitation": 8,

        "washroom": 7,
        "urinal": 7,

        "septic tank": 8,
        "septic": 6
    },


    # ========================================================
    # BUILDING
    # ========================================================

    "BUILDING": {

        "school building": 9,

        "community hall": 9,
        "community building": 9,

        "cremation shed": 9,
        "cremation hall": 9,

        "classroom": 8,

        "boundary wall": 8,
        "compound wall": 8,

        "roof shed": 8,

        "building": 6,

        "school": 4,

        "hall": 5,

        "shed": 5,

        "room": 4,

        "library building": 7,

        "library": 5,

        "hostel": 5,

        "shelter": 5
    }
}


# ============================================================
# 4. OTHER SUBTYPE RULES
# ============================================================
#
# This is the major improvement over our earlier classifier.
#
# "OTHER" is not necessarily meaningless.
#
# Your actual dataset contains many types such as:
#
#   laboratory equipment
#   medical equipment
#   vehicles
#   irrigation
#   flood protection
#   sports
#   playgrounds
#   agriculture
#   retaining walls
#
# We preserve them as OTHER at the broad level but assign
# useful subtypes.
#
# ============================================================

OTHER_SUBTYPE_RULES = {

    # --------------------------------------------------------
    # Education / equipment
    # --------------------------------------------------------

    "LAB_EQUIPMENT": [
        "laboratory equipment",
        "lab equipment",
        "science equipment",
        "laboratory"
    ],

    "SCHOOL_EQUIPMENT": [
        "school equipment",
        "teaching equipment",
        "training equipment",
        "computer",
        "printer",
        "projector"
    ],


    # --------------------------------------------------------
    # Medical
    # --------------------------------------------------------

    "MEDICAL_EQUIPMENT": [
        "operation theater equipment",
        "operation theatre equipment",
        "medical equipment",
        "hospital equipment",
        "multipara monitor",
        "diathermy",
        "suction apparatus",
        "bp apparatus"
    ],

    "MEDICAL_VEHICLE": [
        "ambulance",
        "mortuary van",
        "hearse van"
    ],


    # --------------------------------------------------------
    # Vehicles
    # --------------------------------------------------------

    "VEHICLE": [
        "vehicle",
        "motorised vehicle",
        "motorised electric scooter",
        "electric scooter",
        "scooter",
        "motor bike",
        "motorbike",
        "bike",
        "bus"
    ],


    # --------------------------------------------------------
    # Irrigation / water-resource works
    # --------------------------------------------------------

    "IRRIGATION": [
        "irrigation",
        "irrigation channel",
        "irrigation scheme",
        "lift irrigation",
        "canal",
        "channel",
        "distributory",
        "pws scheme",
        "pws",
        "augmentation of pws",
        "water resource"
    ],


    # --------------------------------------------------------
    # Flood / protection
    # --------------------------------------------------------

    "FLOOD_PROTECTION": [
        "flood protection",
        "flood control",
        "protection work",
        "river training",
        "embankment",
        "retaining wall",
        "protection wall",
        "slope protection",
        "surplus weir"
    ],


    # --------------------------------------------------------
    # Sports
    # --------------------------------------------------------

    "SPORTS": [
        "sports",
        "playground",
        "play ground",
        "stadium",
        "badminton",
        "volleyball",
        "football",
        "cricket"
    ],


    # --------------------------------------------------------
    # Agriculture
    # --------------------------------------------------------

    "AGRICULTURE": [
        "agriculture",
        "agricultural",
        "fishery",
        "fisheries",
        "sericulture",
        "plantation",
        "orange trees",
        "farm"
    ],


    # --------------------------------------------------------
    # Public spaces / parks
    # --------------------------------------------------------

    "PARK_PUBLIC_SPACE": [
        "public park",
        "park development",
        "garden",
        "play area",
        "community playground"
    ],


    # --------------------------------------------------------
    # Footpath
    # --------------------------------------------------------

    "FOOTPATH": [
        "footpath",
        "foot path",
        "foot track",
        "porter track",
        "walkway"
    ],


    # --------------------------------------------------------
    # Shelter
    # --------------------------------------------------------

    "SHELTER": [
        "bus shelter",
        "shelter",
        "canopy",
        "rest house"
    ],


    # --------------------------------------------------------
    # Furniture / fixtures
    # --------------------------------------------------------

    "FURNITURE": [
        "furniture",
        "desks",
        "chairs",
        "table",
        "bench"
    ],


    # --------------------------------------------------------
    # Civil structures
    # --------------------------------------------------------

    "CIVIL_STRUCTURE": [
        "retaining structure",
        "compound structure",
        "gate",
        "welcome gate",
        "fencing",
        "chain link fencing",
        "platform",
        "chabutra",
        "mandapam"
    ]
}


# ============================================================
# 5. CLASSIFY BROAD WORK TYPE
# ============================================================

def classify_work_type(text):
    """
    Return:

        primary_type
        secondary_type
        confidence
        match_count
        scores
    """

    if not isinstance(text, str):
        return "OTHER", None, 0.0, 0, {}

    text = text.lower().strip()

    if not text:
        return "OTHER", None, 0.0, 0, {}


    # --------------------------------------------------------
    # Calculate score for every broad type
    # --------------------------------------------------------

    scores = {
        work_type: 0
        for work_type in WORK_TYPES
        if work_type != "OTHER"
    }


    for work_type, rules in WORK_TYPE_RULES.items():

        for phrase, weight in rules.items():

            if phrase in text:
                scores[work_type] += weight


    # --------------------------------------------------------
    # Keep only matched types
    # --------------------------------------------------------

    matched = {
        work_type: score
        for work_type, score in scores.items()
        if score > 0
    }


    # --------------------------------------------------------
    # Nothing matched
    # --------------------------------------------------------

    if not matched:

        return (
            "OTHER",
            None,
            0.0,
            0,
            {}
        )


    # --------------------------------------------------------
    # Rank the matches
    # --------------------------------------------------------

    ranked = sorted(
        matched.items(),
        key=lambda x: x[1],
        reverse=True
    )


    primary_type = ranked[0][0]
    primary_score = ranked[0][1]


    # --------------------------------------------------------
    # Find secondary type
    # --------------------------------------------------------

    secondary_type = None

    if len(ranked) >= 2:

        second_type = ranked[1][0]
        second_score = ranked[1][1]

        # Only keep it if the second signal is meaningful.
        #
        # Example:
        #
        # CC road + drainage
        #
        # ROAD and DRAINAGE can both be preserved.

        if second_score >= 4:

            secondary_type = second_type

    else:

        second_score = 0


    # --------------------------------------------------------
    # Confidence
    # --------------------------------------------------------

    if primary_score <= 0:

        confidence = 0.0

    else:

        score_gap = primary_score - second_score

        confidence = (
            0.60 +
            0.05 * score_gap
        )

        confidence = min(
            confidence,
            0.95
        )


    # If multiple significant matches exist,
    # reduce confidence slightly.

    if secondary_type is not None:

        confidence *= 0.85


    confidence = round(
        confidence,
        3
    )


    return (
        primary_type,
        secondary_type,
        confidence,
        len(matched),
        matched
    )


# ============================================================
# 6. CLASSIFY OTHER SUBTYPE
# ============================================================

def classify_other_subtype(text):
    """
    Classify records whose broad type is OTHER.

    Returns:
        subtype
        score
    """

    if not isinstance(text, str):
        return "OTHER", 0

    text = text.lower().strip()

    if not text:
        return "OTHER", 0


    subtype_scores = {}


    for subtype, phrases in OTHER_SUBTYPE_RULES.items():

        score = 0

        for phrase in phrases:

            if phrase in text:

                # Longer phrases give stronger evidence.
                words = len(
                    phrase.split()
                )

                score += (
                    2 + words
                )

        if score > 0:

            subtype_scores[subtype] = score


    if not subtype_scores:

        return "OTHER", 0


    subtype, score = max(
        subtype_scores.items(),
        key=lambda x: x[1]
    )


    return subtype, score


# ============================================================
# 7. APPLY BROAD CLASSIFICATION
# ============================================================

classification_results = (

    df["work_description_model"]
    .apply(classify_work_type)

)


# ============================================================
# 8. CREATE BROAD CLASSIFICATION COLUMNS
# ============================================================

df["primary_work_type"] = (

    classification_results
    .apply(lambda x: x[0])

)

df["secondary_work_type"] = (

    classification_results
    .apply(lambda x: x[1])

)

df["work_type_confidence"] = (

    classification_results
    .apply(lambda x: x[2])

)

df["work_type_match_count"] = (

    classification_results
    .apply(lambda x: x[3])

)

df["work_type_scores"] = (

    classification_results
    .apply(lambda x: x[4])

)


# ============================================================
# 9. CLASSIFY SUBTYPE
# ============================================================

def get_subtype(row):

    text = row["work_description_model"]

    primary_type = row["primary_work_type"]

    # --------------------------------------------------------
    # ROAD subtypes
    # --------------------------------------------------------

    if primary_type == "ROAD":

        if "paver block" in text or "paver road" in text:

            return "PAVER_ROAD"

        if "interlock" in text:

            return "INTERLOCK_ROAD"

        if "pcc road" in text or "cc road" in text:

            if (
                "drain" in text
                or "drainage" in text
                or "nali" in text
            ):
                return "CC_ROAD_WITH_DRAINAGE"

            return "CC_ROAD"

        if "bt road" in text or "black top" in text:

            return "BT_ROAD"

        if "tar road" in text:

            return "TAR_ROAD"

        return "OTHER_ROAD"


    # --------------------------------------------------------
    # WATER subtypes
    # --------------------------------------------------------

    if primary_type == "WATER":

        if (
            "borewell" in text
            or "bore well" in text
        ):

            if "pump" in text:

                return "BOREWELL_PUMP"

            return "BOREWELL"

        if (
            "ro system" in text
            or "ro plant" in text
            or "mineral water plant" in text
        ):

            return "WATER_PURIFICATION"

        if (
            "hand pump" in text
            or "handpump" in text
        ):

            return "HANDPUMP"

        if (
            "water tank" in text
            or "water tanker" in text
        ):

            return "WATER_TANK"

        if (
            "pipeline" in text
            or "pipe line" in text
        ):

            return "WATER_PIPELINE"

        if "water harvesting" in text:

            return "WATER_HARVESTING"

        if "pump" in text:

            return "PUMP"

        return "OTHER_WATER"


    # --------------------------------------------------------
    # DRAINAGE subtypes
    # --------------------------------------------------------

    if primary_type == "DRAINAGE":

        if (
            "storm water" in text
            or "storm drain" in text
        ):

            return "STORM_DRAIN"

        if "box drain" in text:

            return "BOX_DRAIN"

        if "side drain" in text:

            return "SIDE_DRAIN"

        if "underground" in text:

            return "UNDERGROUND_DRAINAGE"

        if "culvert" in text:

            return "CULVERT"

        return "OTHER_DRAINAGE"


    # --------------------------------------------------------
    # ELECTRIFICATION subtypes
    # --------------------------------------------------------

    if primary_type == "ELECTRIFICATION":

        if "high mast" in text:

            return "HIGH_MAST"

        if "solar street light" in text:

            return "SOLAR_STREET_LIGHT"

        if "solar light" in text:

            return "SOLAR_LIGHT"

        if (
            "street light" in text
            or "streetlight" in text
        ):

            return "STREET_LIGHT"

        if "transformer" in text:

            return "TRANSFORMER"

        if (
            "11 kv line" in text
            or "ht line" in text
            or "lt line" in text
            or "power line" in text
            or "electric line" in text
        ):

            return "ELECTRICAL_LINE"

        if (
            "solar power plant" in text
        ):

            return "SOLAR_POWER"

        return "OTHER_ELECTRIFICATION"


    # --------------------------------------------------------
    # SANITATION subtypes
    # --------------------------------------------------------

    if primary_type == "SANITATION":

        if (
            "school toilet" in text
        ):

            return "SCHOOL_TOILET"

        if (
            "community toilet" in text
            or "public toilet" in text
        ):

            return "PUBLIC_TOILET"

        if "septic" in text:

            return "SEPTIC"

        if "urinal" in text:

            return "URINAL"

        return "OTHER_SANITATION"


    # --------------------------------------------------------
    # BUILDING subtypes
    # --------------------------------------------------------

    if primary_type == "BUILDING":

        if "school" in text:

            return "SCHOOL"

        if "community hall" in text:

            return "COMMUNITY_HALL"

        if (
            "boundary wall" in text
            or "compound wall" in text
        ):

            return "BOUNDARY_WALL"

        if "classroom" in text:

            return "CLASSROOM"

        if "cremation" in text:

            return "CREMATION_STRUCTURE"

        if "library" in text:

            return "LIBRARY"

        if "hostel" in text:

            return "HOSTEL"

        if "shed" in text:

            return "SHED"

        if "shelter" in text:

            return "SHELTER"

        return "OTHER_BUILDING"


    # --------------------------------------------------------
    # OTHER
    # --------------------------------------------------------

    subtype, _ = classify_other_subtype(text)

    return subtype


df["work_subtype"] = (
    df.apply(
        get_subtype,
        axis=1
    )
)


# ============================================================
# 10. REMOVE TEMPORARY CLASSIFICATION COLUMN?
# ============================================================
#
# Keep work_description_model because it is useful for
# debugging and later NLP.
#
# Do NOT remove it yet.
#
# ============================================================


# ============================================================
# 11. DISPLAY BROAD DISTRIBUTION
# ============================================================

print("\n")
print("=" * 75)
print("PRIMARY WORK TYPE DISTRIBUTION")
print("=" * 75)

type_counts = (

    df["primary_work_type"]
    .value_counts()

)

print(type_counts)


# ============================================================
# 12. DISPLAY PERCENTAGES
# ============================================================

print("\n")
print("=" * 75)
print("PRIMARY WORK TYPE PERCENTAGES")
print("=" * 75)

type_percentages = (

    df["primary_work_type"]
    .value_counts(
        normalize=True
    )
    .mul(100)
    .round(2)

)

print(type_percentages)


# ============================================================
# 13. SUBTYPE DISTRIBUTION
# ============================================================

print("\n")
print("=" * 75)
print("WORK SUBTYPE DISTRIBUTION")
print("=" * 75)

subtype_counts = (

    df["work_subtype"]
    .value_counts()

)

print(
    subtype_counts.head(50)
)


# ============================================================
# 14. CONFIDENCE DISTRIBUTION
# ============================================================

print("\n")
print("=" * 75)
print("CONFIDENCE DISTRIBUTION")
print("=" * 75)

print(
    df["work_type_confidence"]
    .describe()
)


# ============================================================
# 15. LOW-CONFIDENCE RECORDS
# ============================================================

low_confidence = (

    df[
        df["work_type_confidence"] < 0.60
    ][
        [
            "work_id",
            "work_description",
            "primary_work_type",
            "secondary_work_type",
            "work_subtype",
            "work_type_confidence",
            "work_type_match_count"
        ]
    ]

)

print("\n")
print("=" * 75)
print("LOW-CONFIDENCE RECORDS")
print("=" * 75)

print(
    "Count:",
    len(low_confidence)
)

print(
    low_confidence
    .head(100)
    .to_string(index=False)
)


# ============================================================
# 16. OTHER ANALYSIS
# ============================================================

other_records = (

    df[
        df["primary_work_type"] == "OTHER"
    ][
        [
            "work_id",
            "work_description",
            "work_subtype",
            "category",
            "state",
            "geo_district"
        ]
    ]

)

print("\n")
print("=" * 75)
print("OTHER WORK ANALYSIS")
print("=" * 75)

print(
    "Total OTHER:",
    len(other_records)
)


print("\nTop OTHER subtypes:")

print(
    df[
        df["primary_work_type"] == "OTHER"
    ]["work_subtype"]
    .value_counts()
    .head(30)
)


# ============================================================
# 17. SAMPLE FOR EVERY BROAD TYPE
# ============================================================

print("\n")
print("=" * 75)
print("SAMPLE RECORDS PER PRIMARY TYPE")
print("=" * 75)


for work_type in WORK_TYPES:

    subset = df[
        df["primary_work_type"] == work_type
    ]

    print("\n")
    print("-" * 75)
    print(
        work_type,
        "| Count =",
        len(subset)
    )
    print("-" * 75)

    if len(subset) == 0:
        continue

    sample_size = min(
        10,
        len(subset)
    )

    sample = subset[
        [
            "work_id",
            "work_description",
            "primary_work_type",
            "secondary_work_type",
            "work_subtype",
            "work_type_confidence"
        ]
    ].sample(
        sample_size,
        random_state=42
    )

    print(
        sample.to_string(index=False)
    )


# ============================================================
# 18. CHECK POTENTIAL MULTI-TYPE WORKS
# ============================================================

multi_type = df[
    df["secondary_work_type"].notna()
][
    [
        "work_id",
        "work_description",
        "primary_work_type",
        "secondary_work_type",
        "work_subtype",
        "work_type_confidence"
    ]
]

print("\n")
print("=" * 75)
print("MULTI-TYPE WORKS")
print("=" * 75)

print(
    "Count:",
    len(multi_type)
)

print(
    multi_type
    .head(100)
    .to_string(index=False)
)


# ============================================================
# 19. SAVE STEP-2 DATASET
# ============================================================

OUTPUT_DIR = Path(
    "artha_pipeline"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


OUTPUT_FILE = (

    OUTPUT_DIR /
    "combined_real_with_work_type.csv"

)


df.to_csv(
    OUTPUT_FILE,
    index=False
)


# ============================================================
# 20. SAVE WORK-TYPE SUMMARY
# ============================================================

work_type_summary = (

    df.groupby(
        "primary_work_type"
    )
    .agg(
        record_count=(
            "work_id",
            "count"
        ),

        average_confidence=(
            "work_type_confidence",
            "mean"
        ),

        median_confidence=(
            "work_type_confidence",
            "median"
        )
    )
    .reset_index()

)


work_type_summary["percentage"] = (

    work_type_summary["record_count"]
    / len(df)
    * 100

).round(2)


work_type_summary.to_csv(
    OUTPUT_DIR /
    "work_type_summary.csv",
    index=False
)


# ============================================================
# 21. SAVE SUBTYPE SUMMARY
# ============================================================

subtype_summary = (

    df.groupby(
        [
            "primary_work_type",
            "work_subtype"
        ]
    )
    .size()
    .reset_index(
        name="record_count"
    )
    .sort_values(
        "record_count",
        ascending=False
    )

)


subtype_summary.to_csv(
    OUTPUT_DIR /
    "work_subtype_summary.csv",
    index=False
)


# ============================================================
# 22. SAVE LOW-CONFIDENCE RECORDS
# ============================================================

low_confidence.to_csv(
    OUTPUT_DIR /
    "low_confidence_work_types.csv",
    index=False
)


# ============================================================
# 23. FINAL SUMMARY
# ============================================================

print("\n")
print("=" * 75)
print("STEP 2 COMPLETE")
print("=" * 75)

print(
    "Input records:",
    len(df)
)

print(
    "Output:",
    OUTPUT_FILE
)

print(
    "\nCreated columns:"
)

print(
    "primary_work_type"
)

print(
    "secondary_work_type"
)

print(
    "work_subtype"
)

print(
    "work_type_confidence"
)

print(
    "work_type_match_count"
)

print(
    "work_type_scores"
)

print(
    "\nFinal primary type distribution:"
)

print(
    df["primary_work_type"]
    .value_counts()
)

print(
    "\nTop subtypes:"
)

print(
    df["work_subtype"]
    .value_counts()
    .head(25)
)

STEP 2 — WORK TYPE CLASSIFICATION

Input records: 23105


PRIMARY WORK TYPE DISTRIBUTION
primary_work_type
ELECTRIFICATION    6153
OTHER              5677
ROAD               4807
BUILDING           3691
WATER              1985
DRAINAGE            654
SANITATION          138
Name: count, dtype: int64


PRIMARY WORK TYPE PERCENTAGES
primary_work_type
ELECTRIFICATION    26.63
OTHER              24.57
ROAD               20.81
BUILDING           15.97
WATER               8.59
DRAINAGE            2.83
SANITATION          0.60
Name: proportion, dtype: float64


WORK SUBTYPE DISTRIBUTION
work_subtype
OTHER                    4749
HIGH_MAST                3210
CC_ROAD                  2074
OTHER_ROAD               1862
SCHOOL                   1678
SOLAR_LIGHT              1031
OTHER_ELECTRIFICATION     993
WATER_TANK                634
OTHER_BUILDING            569
STREET_LIGHT              554
COMMUNITY_HALL            519
OTHER_DRAINAGE            512
SHED                      474
PAVER_ROAD



PRIMARY WORK TYPE DISTRIBUTION
primary_work_type
ELECTRIFICATION    6153
OTHER              5677
ROAD               4807
BUILDING           3691
WATER              1985
DRAINAGE            654
SANITATION          138
Name: count, dtype: int64


PRIMARY WORK TYPE PERCENTAGES
primary_work_type
ELECTRIFICATION    26.63
OTHER              24.57
ROAD               20.81
BUILDING           15.97
WATER               8.59
DRAINAGE            2.83
SANITATION          0.60
Name: proportion, dtype: float64


WORK SUBTYPE DISTRIBUTION
work_subtype
OTHER                    4749
HIGH_MAST                3210
CC_ROAD                  2074
OTHER_ROAD               1862
SCHOOL                   1678
SOLAR_LIGHT              1031
OTHER_ELECTRIFICATION     993
WATER_TANK                634
OTHER_BUILDING            569
STREET_LIGHT              554
COMMUNITY_HALL            519
OTHER_DRAINAGE            512
SHED                      474
PAVER_ROAD                406
HANDPUMP                  391
SOLAR_



STEP 2 COMPLETE
Input records: 23105
Output: artha_pipeline/combined_real_with_work_type.csv

Created columns:
primary_work_type
secondary_work_type
work_subtype
work_type_confidence
work_type_match_count
work_type_scores

Final primary type distribution:
primary_work_type
ELECTRIFICATION    6153
OTHER              5677
ROAD               4807
BUILDING           3691
WATER              1985
DRAINAGE            654
SANITATION          138
Name: count, dtype: int64

Top subtypes:
work_subtype
OTHER                    4749
HIGH_MAST                3210
CC_ROAD                  2074
OTHER_ROAD               1862
SCHOOL                   1678
SOLAR_LIGHT              1031
OTHER_ELECTRIFICATION     993
WATER_TANK                634
OTHER_BUILDING            569
STREET_LIGHT              554
COMMUNITY_HALL            519
OTHER_DRAINAGE            512
SHED                      474
PAVER_ROAD                406
HANDPUMP                  391
SOLAR_STREET_LIGHT        330
OTHER_WATER           

#step 3 : Which other works should this project be compared against

In [31]:
# ============================================================
# STEP 3 — SUBTYPE-AWARE PEER GROUP CONSTRUCTION
#
# Input  : df  (output from Step 2)
# Output : peer_df
#
# Minimum peer size = 15
#
# Hierarchy:
#
# Level 1:
# WORK TYPE + SUBTYPE + CATEGORY + STATE + DISTRICT + YEAR
#
# Level 2:
# WORK TYPE + SUBTYPE + STATE + DISTRICT + YEAR
#
# Level 3:
# WORK TYPE + SUBTYPE + STATE + DISTRICT
#
# Level 4:
# WORK TYPE + STATE + DISTRICT + YEAR
#
# Level 5:
# WORK TYPE + STATE + DISTRICT
#
# Level 6:
# WORK TYPE + STATE
#
# Level 7:
# WORK TYPE
#
# The first level with >= 15 works is selected.
# ============================================================

import pandas as pd
import numpy as np


# ============================================================
# 1. CREATE A SEPARATE DATAFRAME
# ============================================================

peer_df = df.copy()

print("========== STEP 3 START ==========")
print("Total works:", len(peer_df))


# ============================================================
# 2. CHECK REQUIRED COLUMNS
# ============================================================

required_columns = [
    "work_id",
    "primary_work_type",
    "work_subtype",
    "category",
    "state",
    "geo_district",
    "completion_year"
]

missing_columns = [
    col
    for col in required_columns
    if col not in peer_df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )


# ============================================================
# 3. NORMALIZE PEER-GROUP FIELDS
# ============================================================

for col in [
    "primary_work_type",
    "work_subtype",
    "category",
    "state",
    "geo_district"
]:

    peer_df[col] = (
        peer_df[col]
        .fillna("UNKNOWN")
        .astype(str)
        .str.strip()
        .str.upper()
    )


# ------------------------------------------------------------
# Completion year
# ------------------------------------------------------------

peer_df["completion_year"] = pd.to_numeric(
    peer_df["completion_year"],
    errors="coerce"
)


# ------------------------------------------------------------
# Create safe year field for grouping
#
# UNKNOWN_YEAR is used instead of losing records with
# missing completion year.
# ------------------------------------------------------------

peer_df["peer_year"] = (
    peer_df["completion_year"]
    .fillna(-1)
    .astype(int)
    .astype(str)
)


# ============================================================
# 4. MINIMUM PEER SIZE
# ============================================================

MIN_PEERS = 15


# ============================================================
# 5. DEFINE SUBTYPE-AWARE PEER HIERARCHY
# ============================================================

peer_levels = {

    # Most specific
    1: [
        "primary_work_type",
        "work_subtype",
        "category",
        "state",
        "geo_district",
        "peer_year"
    ],

    # Remove category
    2: [
        "primary_work_type",
        "work_subtype",
        "state",
        "geo_district",
        "peer_year"
    ],

    # Remove year
    3: [
        "primary_work_type",
        "work_subtype",
        "state",
        "geo_district"
    ],

    # Fall back to broad work type
    4: [
        "primary_work_type",
        "state",
        "geo_district",
        "peer_year"
    ],

    5: [
        "primary_work_type",
        "state",
        "geo_district"
    ],

    6: [
        "primary_work_type",
        "state"
    ],

    7: [
        "primary_work_type"
    ]
}


# ============================================================
# 6. CALCULATE CANDIDATE GROUP SIZE FOR EACH LEVEL
# ============================================================

level_sizes = {}

print("\n========== BUILDING CANDIDATE PEER GROUPS ==========")

for level, group_columns in peer_levels.items():

    size_table = (
        peer_df
        .groupby(
            group_columns,
            dropna=False
        )
        .size()
        .reset_index(
            name=f"level_{level}_size"
        )
    )

    level_sizes[level] = size_table

    print(
        f"Level {level}: "
        f"{len(size_table):,} candidate groups"
    )


# ============================================================
# 7. ATTACH ALL LEVEL SIZES TO EACH WORK
# ============================================================

for level, size_table in level_sizes.items():

    group_columns = peer_levels[level]

    peer_df = peer_df.merge(
        size_table,
        on=group_columns,
        how="left",
        validate="many_to_one"
    )


# ============================================================
# 8. SELECT THE MOST SPECIFIC VALID LEVEL
#
# Start at Level 7 as the fallback.
# Then replace with a more specific level whenever that
# level has >= MIN_PEERS.
# ============================================================

peer_df["peer_level"] = 7

for level in range(1, 7):

    size_column = f"level_{level}_size"

    valid = (
        peer_df[size_column]
        >= MIN_PEERS
    )

    peer_df.loc[
        valid,
        "peer_level"
    ] = np.minimum(
        peer_df.loc[valid, "peer_level"],
        level
    )


# ============================================================
# 9. BUILD PEER GROUP DEFINITION
# ============================================================

peer_df["peer_group_definition"] = ""

for level, group_columns in peer_levels.items():

    mask = (
        peer_df["peer_level"] == level
    )

    peer_df.loc[
        mask,
        "peer_group_definition"
    ] = " + ".join(group_columns)


# ============================================================
# 10. ASSIGN PEER GROUP SIZE
# ============================================================

peer_df["peer_group_size"] = np.nan

for level in range(1, 8):

    size_column = f"level_{level}_size"

    mask = (
        peer_df["peer_level"] == level
    )

    peer_df.loc[
        mask,
        "peer_group_size"
    ] = peer_df.loc[
        mask,
        size_column
    ]


peer_df["peer_group_size"] = (
    peer_df["peer_group_size"]
    .astype(int)
)


# ============================================================
# 11. CREATE A UNIQUE PEER GROUP ID
#
# The level is included so that the same textual combination
# at different hierarchy levels can never be confused.
# ============================================================

peer_df["peer_group_id"] = ""

for level, group_columns in peer_levels.items():

    mask = (
        peer_df["peer_level"] == level
    )

    def build_peer_id(row):

        values = []

        for col in group_columns:

            values.append(
                f"{col}={row[col]}"
            )

        return (
            f"L{level} | "
            + " | ".join(values)
        )

    peer_df.loc[mask, "peer_group_id"] = (
        peer_df.loc[mask]
        .apply(
            build_peer_id,
            axis=1
        )
    )


# ============================================================
# 12. REMOVE TEMPORARY COLUMNS
# ============================================================

temporary_columns = [
    "peer_year"
]

for level in range(1, 8):

    temporary_columns.append(
        f"level_{level}_size"
    )

peer_df.drop(
    columns=temporary_columns,
    inplace=True,
    errors="ignore"
)


# ============================================================
# 13. FINAL TYPE CLEANUP
# ============================================================

peer_df["peer_level"] = (
    peer_df["peer_level"]
    .astype(int)
)

peer_df["peer_group_size"] = (
    peer_df["peer_group_size"]
    .astype(int)
)


# ============================================================
# 14. BASIC VALIDATION
# ============================================================

print("\n========== STEP 3 VALIDATION ==========")

print(
    "Total works:",
    len(peer_df)
)

print(
    "Missing peer IDs:",
    peer_df["peer_group_id"].isna().sum()
)

print(
    "Missing peer levels:",
    peer_df["peer_level"].isna().sum()
)

print(
    "Missing peer sizes:",
    peer_df["peer_group_size"].isna().sum()
)

print(
    "Minimum peer-group size:",
    peer_df["peer_group_size"].min()
)

print(
    "Maximum peer-group size:",
    peer_df["peer_group_size"].max()
)

print(
    "Works with peer size <",
    MIN_PEERS,
    ":",
    (
        peer_df["peer_group_size"]
        < MIN_PEERS
    ).sum()
)


# ============================================================
# 15. PEER LEVEL DISTRIBUTION
# ============================================================

level_distribution = (
    peer_df["peer_level"]
    .value_counts()
    .sort_index()
)

level_percentage = (
    peer_df["peer_level"]
    .value_counts(
        normalize=True
    )
    .sort_index()
    .mul(100)
    .round(2)
)

print(
    "\n========== PEER LEVEL DISTRIBUTION =========="
)

print(level_distribution)

print("\nPercentage:")
print(level_percentage)


# ============================================================
# 16. PEER GROUP SIZE SUMMARY
# ============================================================

peer_size_summary = (
    peer_df["peer_group_size"]
    .describe()
)

print(
    "\n========== PEER GROUP SIZE =========="
)

print(peer_size_summary)


# ============================================================
# 17. NUMBER OF UNIQUE PEER GROUPS
# ============================================================

unique_peer_groups = (
    peer_df["peer_group_id"]
    .nunique()
)

print(
    "\nUnique peer-group IDs:",
    unique_peer_groups
)


# ============================================================
# 18. CHECK DUPLICATE PEER GROUP DEFINITIONS
# ============================================================

peer_group_check = (
    peer_df[
        [
            "peer_group_id",
            "peer_level",
            "peer_group_size",
            "peer_group_definition"
        ]
    ]
    .drop_duplicates()
)

print(
    "Unique peer-group records:",
    len(peer_group_check)
)


# ============================================================
# 19. SHOW SAMPLE ASSIGNMENTS
# ============================================================

print(
    "\n========== SAMPLE PEER ASSIGNMENTS =========="
)

print(
    peer_df[
        [
            "work_id",
            "primary_work_type",
            "work_subtype",
            "category",
            "state",
            "geo_district",
            "completion_year",
            "peer_group_id",
            "peer_level",
            "peer_group_size",
            "peer_group_definition"
        ]
    ]
    .sample(
        10,
        random_state=42
    )
)


# ============================================================
# 20. SHOW HOW MANY WORKS USE SUBTYPE-AWARE LEVELS
# ============================================================

subtype_levels = (
    peer_df["peer_level"] <= 3
)

print(
    "\n========== SUBTYPE-AWARE COVERAGE =========="
)

print(
    "Works using subtype-aware peer groups:",
    subtype_levels.sum()
)

print(
    "Percentage using subtype-aware groups:",
    round(
        subtype_levels.mean() * 100,
        2
    ),
    "%"
)


# ============================================================
# 21. FINAL STEP 3 CHECK
# ============================================================

print(
    "\n========== FINAL STEP 3 CHECK =========="
)

checks = {

    "Row count unchanged":
        len(peer_df) == len(df),

    "No missing peer IDs":
        peer_df["peer_group_id"].notna().all(),

    "No missing peer levels":
        peer_df["peer_level"].notna().all(),

    "No missing peer sizes":
        peer_df["peer_group_size"].notna().all(),

    "Minimum peer size satisfied":
        peer_df["peer_group_size"].ge(
            MIN_PEERS
        ).all()

}

all_passed = True

for check_name, result in checks.items():

    print(
        ("✅" if result else "❌"),
        check_name,
        ":",
        result
    )

    if not result:
        all_passed = False


if all_passed:

    print(
        "\n✅ STEP 3 PASSED"
    )

    print(
        "✅ Subtype-aware peer groups created"
    )

    print(
        "✅ All works have >= "
        f"{MIN_PEERS} peers"
    )

else:

    print(
        "\n❌ STEP 3 NEEDS INVESTIGATION"
    )

========== STEP 3 START ==========
Total works: 23105

========== BUILDING CANDIDATE PEER GROUPS ==========
Level 1: 5,546 candidate groups
Level 2: 5,383 candidate groups
Level 3: 4,023 candidate groups
Level 4: 3,630 candidate groups
Level 5: 2,357 candidate groups
Level 6: 207 candidate groups
Level 7: 7 candidate groups

========== STEP 3 VALIDATION ==========
Total works: 23105
Missing peer IDs: 0
Missing peer levels: 0
Missing peer sizes: 0
Minimum peer-group size: 15
Maximum peer-group size: 6153
Works with peer size < 15 : 0

========== PEER LEVEL DISTRIBUTION ==========
peer_level
1    10326
2       57
3     2530
4     1183
5     1245
6     7429
7      335
Name: count, dtype: int64

Percentage:
peer_level
1    44.69
2     0.25
3    10.95
4     5.12
5     5.39
6    32.15
7     1.45
Name: proportion, dtype: float64

========== PEER GROUP SIZE ==========
count    23105.000000
mean       153.485652
std        433.475780
min         15.000000
25%         23.000000
50%         47.00


========== STEP 3 VALIDATION ==========
Total works: 23105
Missing peer IDs: 0
Missing peer levels: 0
Missing peer sizes: 0
Minimum peer-group size: 15
Maximum peer-group size: 6153
Works with peer size < 15 : 0

========== PEER LEVEL DISTRIBUTION ==========
peer_level
1    10326
2       57
3     2530
4     1183
5     1245
6     7429
7      335
Name: count, dtype: int64

Percentage:
peer_level
1    44.69
2     0.25
3    10.95
4     5.12
5     5.39
6    32.15
7     1.45
Name: proportion, dtype: float64

========== PEER GROUP SIZE ==========
count    23105.000000
mean       153.485652
std        433.475780
min         15.000000
25%         23.000000
50%         47.000000
75%        132.000000
max       6153.000000
Name: peer_group_size, dtype: float64

Unique peer-group IDs: 1060
Unique peer-group records: 1060

========== SAMPLE PEER ASSIGNMENTS ==========
       work_id  ...                              peer_group_definition
10049   191316  ...                          primary_work_ty

#step 4: Cost Anomaly Detection.

In [32]:
# ============================================================
# STEP 4 — FINAL COST ANOMALY DETECTION
#
# Input:
#     peer_df  -> validated output of Step 3
#
# Output:
#     cost_df
#
# Detection:
#     1. IQR                  -> PRIMARY SIGNAL
#     2. Stabilized MAD       -> SUPPORTING SIGNAL
#     3. Relative deviation  -> SEVERITY / EVIDENCE ONLY
#
# IMPORTANT:
#     Relative deviation alone CANNOT create an anomaly.
#
#     Final anomaly:
#         IQR anomaly OR MAD anomaly
#
# Score:
#     IQR anomaly       -> 60 points
#     MAD anomaly       -> 30 points
#     Relative severity -> up to 10 points
#
# Maximum score = 100
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# 1. CREATE WORKING COPY
# ============================================================

cost_df = peer_df.copy()

print("============================================================")
print("             STEP 4 — COST ANOMALY DETECTION")
print("============================================================")

print("Total works:", len(cost_df))


# ============================================================
# 2. COST VALIDATION
# ============================================================

cost_df["cost"] = pd.to_numeric(
    cost_df["cost"],
    errors="coerce"
)

valid_cost_mask = (
    cost_df["cost"].notna()
    &
    (cost_df["cost"] > 0)
)

print("\n========== COST VALIDATION ==========")

print(
    "Valid costs:",
    valid_cost_mask.sum()
)

print(
    "Invalid / missing costs:",
    (~valid_cost_mask).sum()
)


# ============================================================
# 3. PREPARE SAME HIERARCHY USED IN STEP 3
# ============================================================

cost_df["peer_year"] = (
    cost_df["completion_year"]
    .fillna(-1)
    .astype(int)
    .astype(str)
)


peer_levels = {

    # --------------------------------------------------------
    # LEVEL 1
    # TYPE + SUBTYPE + CATEGORY + STATE + DISTRICT + YEAR
    # --------------------------------------------------------
    1: [
        "primary_work_type",
        "work_subtype",
        "category",
        "state",
        "geo_district",
        "peer_year"
    ],

    # --------------------------------------------------------
    # LEVEL 2
    # TYPE + SUBTYPE + STATE + DISTRICT + YEAR
    # --------------------------------------------------------
    2: [
        "primary_work_type",
        "work_subtype",
        "state",
        "geo_district",
        "peer_year"
    ],

    # --------------------------------------------------------
    # LEVEL 3
    # TYPE + SUBTYPE + STATE + DISTRICT
    # --------------------------------------------------------
    3: [
        "primary_work_type",
        "work_subtype",
        "state",
        "geo_district"
    ],

    # --------------------------------------------------------
    # LEVEL 4
    # TYPE + STATE + DISTRICT + YEAR
    # --------------------------------------------------------
    4: [
        "primary_work_type",
        "state",
        "geo_district",
        "peer_year"
    ],

    # --------------------------------------------------------
    # LEVEL 5
    # TYPE + STATE + DISTRICT
    # --------------------------------------------------------
    5: [
        "primary_work_type",
        "state",
        "geo_district"
    ],

    # --------------------------------------------------------
    # LEVEL 6
    # TYPE + STATE
    # --------------------------------------------------------
    6: [
        "primary_work_type",
        "state"
    ],

    # --------------------------------------------------------
    # LEVEL 7
    # TYPE
    # --------------------------------------------------------
    7: [
        "primary_work_type"
    ]
}


# ============================================================
# 4. VALIDATE STEP 3 OUTPUT
# ============================================================

print("\n========== STEP 3 OUTPUT VALIDATION ==========")

print(
    "Missing peer_group_id:",
    cost_df["peer_group_id"].isna().sum()
)

print(
    "Missing peer_level:",
    cost_df["peer_level"].isna().sum()
)

print(
    "Missing peer_group_size:",
    cost_df["peer_group_size"].isna().sum()
)

print(
    "Works below minimum peer size:",
    (
        cost_df["peer_group_size"] < 15
    ).sum()
)


# ============================================================
# 5. MAD FUNCTION
# ============================================================

def calculate_mad(series):

    median = series.median()

    return (
        series - median
    ).abs().median()


# ============================================================
# 6. CALCULATE CANDIDATE STATISTICS FOR EACH LEVEL
# ============================================================

level_statistics = {}

print(
    "\n========== BUILDING PEER STATISTICS =========="
)

for level, group_columns in peer_levels.items():

    print(
        f"Calculating Level {level}..."
    )

    # --------------------------------------------------------
    # Basic statistics
    # --------------------------------------------------------

    stats = (
        cost_df
        .groupby(
            group_columns,
            dropna=False
        )["cost"]
        .agg(
            peer_median="median",
            peer_mean="mean",
            peer_q1=lambda x: x.quantile(0.25),
            peer_q3=lambda x: x.quantile(0.75),
            peer_actual_count="count"
        )
        .reset_index()
    )

    # --------------------------------------------------------
    # IQR
    # --------------------------------------------------------

    stats["iqr"] = (
        stats["peer_q3"]
        -
        stats["peer_q1"]
    )

    # --------------------------------------------------------
    # IQR lower/upper boundaries
    # --------------------------------------------------------

    stats["iqr_lower"] = (
        stats["peer_q1"]
        -
        1.5 * stats["iqr"]
    )

    stats["iqr_upper"] = (
        stats["peer_q3"]
        +
        1.5 * stats["iqr"]
    )

    # --------------------------------------------------------
    # MAD
    # --------------------------------------------------------

    mad_stats = (
        cost_df
        .groupby(
            group_columns,
            dropna=False
        )["cost"]
        .apply(calculate_mad)
        .reset_index(
            name="mad"
        )
    )

    stats = stats.merge(
        mad_stats,
        on=group_columns,
        how="left",
        validate="one_to_one"
    )

    # --------------------------------------------------------
    # STABILIZED MAD
    #
    # Prevent extremely tiny MAD values from creating absurd
    # robust-Z values.
    #
    # effective MAD =
    # max(
    #     actual MAD,
    #     25% of IQR,
    #     1% of peer median
    # )
    # --------------------------------------------------------

    mad_component = (
        stats["mad"]
        .fillna(0)
        .to_numpy()
    )

    iqr_component = (
        stats["iqr"]
        .abs()
        .fillna(0)
        .mul(0.25)
        .to_numpy()
    )

    median_component = (
        stats["peer_median"]
        .abs()
        .fillna(0)
        .mul(0.01)
        .to_numpy()
    )

    stats["effective_mad"] = np.maximum.reduce(
        [
            mad_component,
            iqr_component,
            median_component
        ]
    )

    level_statistics[level] = stats

    print(
        f"Level {level}: "
        f"{len(stats):,} candidate groups"
    )


# ============================================================
# 7. INITIALIZE STATISTIC COLUMNS
# ============================================================

statistical_columns = [

    "peer_median",
    "peer_mean",
    "peer_q1",
    "peer_q3",
    "peer_actual_count",
    "iqr",
    "iqr_lower",
    "iqr_upper",
    "mad",
    "effective_mad"
]

for column in statistical_columns:

    cost_df[column] = np.nan


# ============================================================
# 8. ATTACH STATISTICS BASED ON SELECTED PEER LEVEL
# ============================================================

print(
    "\n========== ATTACHING PEER STATISTICS =========="
)

for level, group_columns in peer_levels.items():

    level_mask = (
        cost_df["peer_level"] == level
    )

    if level_mask.sum() == 0:
        continue

    stats = level_statistics[level][
        group_columns
        +
        statistical_columns
    ]

    selected_rows = cost_df.loc[
        level_mask,
        group_columns
    ].copy()

    selected_rows["_original_index"] = (
        selected_rows.index
    )

    selected_rows = selected_rows.merge(
        stats,
        on=group_columns,
        how="left",
        validate="many_to_one"
    )

    selected_rows = selected_rows.set_index(
        "_original_index"
    )

    for column in statistical_columns:

        cost_df.loc[
            selected_rows.index,
            column
        ] = selected_rows[column]


# ============================================================
# 9. VALIDATE STATISTICS
# ============================================================

print(
    "\n========== STATISTICS VALIDATION =========="
)

for column in statistical_columns:

    print(
        f"{column}:",
        cost_df[column].isna().sum(),
        "missing"
    )


# ============================================================
# 10. PEER SIZE VALIDATION
# ============================================================

peer_size_difference = (
    cost_df["peer_group_size"]
    -
    cost_df["peer_actual_count"]
)

print(
    "\n========== PEER SIZE VALIDATION =========="
)

print(
    "Peer size differences:",
    (
        peer_size_difference != 0
    ).sum()
)


# ============================================================
# 11. IQR ANOMALY
#
# PRIMARY SIGNAL
# ============================================================

cost_df["iqr_anomaly"] = (
    (
        cost_df["cost"]
        <
        cost_df["iqr_lower"]
    )
    |
    (
        cost_df["cost"]
        >
        cost_df["iqr_upper"]
    )
)


# ============================================================
# 12. COST / PEER MEDIAN RATIO
#
# Used for interpretation and severity.
# NOT a standalone anomaly detector.
# ============================================================

cost_df["cost_to_peer_median_ratio"] = np.nan

valid_median_mask = (
    cost_df["peer_median"].notna()
    &
    (
        cost_df["peer_median"] > 0
    )
)

cost_df.loc[
    valid_median_mask,
    "cost_to_peer_median_ratio"
] = (
    cost_df.loc[
        valid_median_mask,
        "cost"
    ]
    /
    cost_df.loc[
        valid_median_mask,
        "peer_median"
    ]
)


# ============================================================
# 13. SYMMETRIC RELATIVE DEVIATION
#
# Example:
#
# ratio = 4
# ratio = 0.25
#
# both become deviation = 4
# ============================================================

cost_df["relative_deviation"] = np.nan

cost_df.loc[
    valid_median_mask,
    "relative_deviation"
] = (
    cost_df.loc[
        valid_median_mask,
        "cost_to_peer_median_ratio"
    ]
    .apply(
        lambda x:
        max(x, 1 / x)
        if x > 0
        else np.nan
    )
)


# ============================================================
# 14. RELATIVE SEVERITY
#
# Relative deviation is NOT an anomaly by itself.
#
# We only use it to add up to 10 points to the score.
# ============================================================

cost_df["relative_severity_points"] = 0.0

relative_valid = (
    cost_df["relative_deviation"].notna()
)

# 2x deviation  -> 2 points
# 3x deviation  -> 4 points
# 5x deviation  -> 6 points
# 10x deviation -> 8 points
# 20x+          -> 10 points

cost_df.loc[
    relative_valid,
    "relative_severity_points"
] = np.select(
    [
        cost_df.loc[
            relative_valid,
            "relative_deviation"
        ] >= 20,

        cost_df.loc[
            relative_valid,
            "relative_deviation"
        ] >= 10,

        cost_df.loc[
            relative_valid,
            "relative_deviation"
        ] >= 5,

        cost_df.loc[
            relative_valid,
            "relative_deviation"
        ] >= 3,

        cost_df.loc[
            relative_valid,
            "relative_deviation"
        ] >= 2
    ],
    [
        10,
        8,
        6,
        4,
        2
    ],
    default=0
)


# ============================================================
# 15. STABILIZED ROBUST Z-SCORE
# ============================================================

cost_df["robust_z"] = np.nan

valid_scale_mask = (
    cost_df["effective_mad"].notna()
    &
    (
        cost_df["effective_mad"] > 0
    )
)

cost_df.loc[
    valid_scale_mask,
    "robust_z"
] = (
    0.6745
    *
    (
        cost_df.loc[
            valid_scale_mask,
            "cost"
        ]
        -
        cost_df.loc[
            valid_scale_mask,
            "peer_median"
        ]
    )
    /
    cost_df.loc[
        valid_scale_mask,
        "effective_mad"
    ]
)


# ============================================================
# 16. MAD ANOMALY
#
# SUPPORTING SIGNAL
#
# MAD = 0 itself does NOT create an anomaly.
# ============================================================

cost_df["mad_anomaly"] = (
    cost_df["robust_z"].abs()
    >
    3.5
)

cost_df["mad_anomaly"] = (
    cost_df["mad_anomaly"]
    .fillna(False)
)


# ============================================================
# 17. FINAL COST ANOMALY
#
# IMPORTANT:
# Relative deviation alone CANNOT trigger this flag.
# ============================================================

cost_df["cost_anomaly"] = (
    cost_df["iqr_anomaly"]
    |
    cost_df["mad_anomaly"]
)


# ============================================================
# 18. DETECTOR OVERLAP
# ============================================================

iqr_only = (
    cost_df["iqr_anomaly"]
    &
    ~cost_df["mad_anomaly"]
)

mad_only = (
    ~cost_df["iqr_anomaly"]
    &
    cost_df["mad_anomaly"]
)

both_anomaly = (
    cost_df["iqr_anomaly"]
    &
    cost_df["mad_anomaly"]
)

neither = (
    ~cost_df["iqr_anomaly"]
    &
    ~cost_df["mad_anomaly"]
)


# ============================================================
# 19. FINAL 0–100 SCORE
#
# IQR              = 60
# MAD              = 30
# Relative severity = up to 10
#
# IMPORTANT:
# A large relative deviation without IQR/MAD anomaly
# can increase severity only if the work is already anomalous.
# ============================================================

cost_df["cost_anomaly_score"] = 0.0

# Primary IQR signal
cost_df.loc[
    cost_df["iqr_anomaly"],
    "cost_anomaly_score"
] += 60

# Supporting MAD signal
cost_df.loc[
    cost_df["mad_anomaly"],
    "cost_anomaly_score"
] += 30

# Relative severity is added ONLY to anomalous records
cost_df.loc[
    cost_df["cost_anomaly"],
    "cost_anomaly_score"
] += (
    cost_df.loc[
        cost_df["cost_anomaly"],
        "relative_severity_points"
    ]
)

cost_df["cost_anomaly_score"] = (
    cost_df["cost_anomaly_score"]
    .clip(
        lower=0,
        upper=100
    )
)


# ============================================================
# 20. SCORE BAND
# ============================================================

def score_band(score):

    if score >= 80:
        return "Strong anomaly signal"

    elif score >= 60:
        return "High anomaly signal"

    elif score >= 40:
        return "Moderate anomaly signal"

    elif score > 0:
        return "Weak anomaly signal"

    else:
        return "No cost anomaly detected"


cost_df["anomaly_score_band"] = (
    cost_df["cost_anomaly_score"]
    .apply(score_band)
)


# ============================================================
# 21. HUMAN-READABLE REASON
# ============================================================

def anomaly_reason(row):

    reasons = []

    if row["iqr_anomaly"]:
        reasons.append(
            "Cost outside peer IQR range"
        )

    if row["mad_anomaly"]:
        reasons.append(
            "Large robust deviation from peer median"
        )

    if (
        row["cost_anomaly"]
        and
        pd.notna(row["relative_deviation"])
        and
        row["relative_deviation"] >= 2
    ):
        reasons.append(
            "Cost differs from peer median by at least 2x"
        )

    if not reasons:
        return "No cost anomaly detected"

    return " + ".join(reasons)


cost_df["anomaly_reason"] = (
    cost_df.apply(
        anomaly_reason,
        axis=1
    )
)


# ============================================================
# 22. REVIEW STATUS
# ============================================================

cost_df["review_status"] = np.where(
    cost_df["cost_anomaly"],
    "Potential anomaly requiring human review",
    "Within expected peer cost range"
)


# ============================================================
# 23. FINAL SUMMARY
# ============================================================

print("\n")
print("============================================================")
print("              STEP 4 — FINAL SUMMARY")
print("============================================================")

print(
    "Total works:",
    len(cost_df)
)

print(
    "IQR anomalies:",
    cost_df["iqr_anomaly"].sum()
)

print(
    "MAD anomalies:",
    cost_df["mad_anomaly"].sum()
)

print(
    "Combined cost anomalies:",
    cost_df["cost_anomaly"].sum()
)

print(
    "Cost anomaly percentage:",
    round(
        cost_df["cost_anomaly"].mean() * 100,
        2
    ),
    "%"
)


# ============================================================
# 24. DETECTOR OVERLAP
# ============================================================

print(
    "\n========== DETECTOR OVERLAP =========="
)

print(
    "IQR only:",
    iqr_only.sum()
)

print(
    "MAD only:",
    mad_only.sum()
)

print(
    "Both:",
    both_anomaly.sum()
)

print(
    "Neither:",
    neither.sum()
)

print(
    "Consistency check:",
    (
        cost_df["cost_anomaly"]
        ==
        (
            cost_df["iqr_anomaly"]
            |
            cost_df["mad_anomaly"]
        )
    ).all()
)


# ============================================================
# 25. SCORE DISTRIBUTION
# ============================================================

print(
    "\n========== SCORE DISTRIBUTION =========="
)

print(
    cost_df[
        "cost_anomaly_score"
    ]
    .value_counts()
    .sort_index()
)


# ============================================================
# 26. SCORE BAND DISTRIBUTION
# ============================================================

print(
    "\n========== SCORE BAND DISTRIBUTION =========="
)

print(
    cost_df[
        "anomaly_score_band"
    ]
    .value_counts()
)


# ============================================================
# 27. MAD INFORMATION
# ============================================================

print(
    "\n========== MAD INFORMATION =========="
)

print(
    "Works with original MAD = 0:",
    (
        cost_df["mad"] == 0
    ).sum()
)

print(
    "Robust Z available:",
    cost_df["robust_z"].notna().sum()
)

print(
    "MAD anomalies:",
    cost_df["mad_anomaly"].sum()
)

print(
    "Maximum absolute robust Z:",
    cost_df["robust_z"]
    .abs()
    .max()
)


# ============================================================
# 28. RELATIVE DEVIATION INFORMATION
# ============================================================

print(
    "\n========== RELATIVE COST INFORMATION =========="
)

print(
    "Median cost-to-peer ratio:",
    cost_df[
        "cost_to_peer_median_ratio"
    ].median()
)

print(
    "Maximum cost-to-peer ratio:",
    cost_df[
        "cost_to_peer_median_ratio"
    ].max()
)

print(
    "Works with >= 2x relative deviation:",
    (
        cost_df["relative_deviation"] >= 2
    ).sum()
)

print(
    "IMPORTANT: relative deviation alone does NOT create "
    "cost_anomaly."
)


# ============================================================
# 29. PEER GROUP SIZE
# ============================================================

print(
    "\n========== PEER GROUP SIZE =========="
)

print(
    cost_df[
        "peer_group_size"
    ].describe()
)


# ============================================================
# 30. ANOMALIES BY WORK TYPE
# ============================================================

print(
    "\n========== ANOMALIES BY WORK TYPE =========="
)

anomaly_by_type = (
    cost_df
    .groupby("primary_work_type")
    .agg(
        total_works=(
            "work_id",
            "count"
        ),
        anomalies=(
            "cost_anomaly",
            "sum"
        )
    )
)

anomaly_by_type[
    "anomaly_percentage"
] = (
    anomaly_by_type["anomalies"]
    /
    anomaly_by_type["total_works"]
    * 100
)

anomaly_by_type[
    "anomaly_percentage"
] = (
    anomaly_by_type[
        "anomaly_percentage"
    ]
    .round(2)
)

print(
    anomaly_by_type
    .sort_values(
        "anomalies",
        ascending=False
    )
)


# ============================================================
# 31. ANOMALIES BY PEER LEVEL
# ============================================================

print(
    "\n========== ANOMALIES BY PEER LEVEL =========="
)

anomaly_by_level = (
    cost_df
    .groupby("peer_level")
    .agg(
        total_works=(
            "work_id",
            "count"
        ),
        anomalies=(
            "cost_anomaly",
            "sum"
        )
    )
)

anomaly_by_level[
    "anomaly_percentage"
] = (
    anomaly_by_level["anomalies"]
    /
    anomaly_by_level["total_works"]
    * 100
)

anomaly_by_level[
    "anomaly_percentage"
] = (
    anomaly_by_level[
        "anomaly_percentage"
    ]
    .round(2)
)

print(
    anomaly_by_level
)


# ============================================================
# 32. TOP ANOMALIES
# ============================================================

top_anomalies = (
    cost_df[
        cost_df["cost_anomaly"]
    ]
    .sort_values(
        [
            "cost_anomaly_score",
            "relative_deviation",
            "robust_z"
        ],
        ascending=[
            False,
            False,
            False
        ]
    )
)

print(
    "\n========== TOP 20 COST ANOMALIES =========="
)

print(
    top_anomalies[
        [
            "work_id",
            "primary_work_type",
            "work_subtype",
            "state",
            "geo_district",
            "completion_year",
            "cost",
            "peer_level",
            "peer_group_size",
            "peer_actual_count",
            "peer_median",
            "peer_q1",
            "peer_q3",
            "iqr",
            "iqr_lower",
            "iqr_upper",
            "mad",
            "effective_mad",
            "robust_z",
            "cost_to_peer_median_ratio",
            "relative_deviation",
            "iqr_anomaly",
            "mad_anomaly",
            "cost_anomaly_score",
            "anomaly_score_band",
            "anomaly_reason",
            "review_status"
        ]
    ]
    .head(20)
)


# ============================================================
# 33. FINAL DATAFRAME INFO
# ============================================================

print(
    "\n========== FINAL COST_DF INFO =========="
)

print(
    cost_df.info()
)


# ============================================================
# 34. FINAL STATUS
# ============================================================

print(
    "\n========== STEP 4 COMPLETE =========="
)

print(
    "✅ IQR = primary anomaly detector"
)

print(
    "✅ Stabilized MAD = supporting detector"
)

print(
    "✅ MAD = 0 does NOT automatically create anomaly"
)

print(
    "✅ Relative cost = severity/evidence only"
)

print(
    "✅ Relative cost alone cannot create anomaly"
)

print(
    "✅ Final anomaly score = 0–100"
)

print(
    "✅ Human-review status created"
)

             STEP 4 — COST ANOMALY DETECTION
Total works: 23105

========== COST VALIDATION ==========
Valid costs: 23105
Invalid / missing costs: 0

========== STEP 3 OUTPUT VALIDATION ==========
Missing peer_group_id: 0
Missing peer_level: 0
Missing peer_group_size: 0
Works below minimum peer size: 0

========== BUILDING PEER STATISTICS ==========
Calculating Level 1...
Level 1: 5,546 candidate groups
Calculating Level 2...
Level 2: 5,383 candidate groups
Calculating Level 3...
Level 3: 4,023 candidate groups
Calculating Level 4...
Level 4: 3,630 candidate groups
Calculating Level 5...
Level 5: 2,357 candidate groups
Calculating Level 6...
Level 6: 207 candidate groups
Calculating Level 7...
Level 7: 7 candidate groups

========== ATTACHING PEER STATISTICS ==========

========== STATISTICS VALIDATION ==========
peer_median: 0 missing
peer_mean: 0 missing
peer_q1: 0 missing
peer_q3: 0 missing
peer_actual_count: 0 missing
iqr: 0 missing
iqr_lower: 0 missing
iqr_upper: 0 missing
mad: 0 

Level 1: 5,546 candidate groups
Calculating Level 2...


Level 2: 5,383 candidate groups
Calculating Level 3...


Level 3: 4,023 candidate groups
Calculating Level 4...


Level 4: 3,630 candidate groups
Calculating Level 5...


Level 5: 2,357 candidate groups
Calculating Level 6...
Level 6: 207 candidate groups
Calculating Level 7...
Level 7: 7 candidate groups

========== ATTACHING PEER STATISTICS ==========

========== STATISTICS VALIDATION ==========
peer_median: 0 missing
peer_mean: 0 missing
peer_q1: 0 missing
peer_q3: 0 missing
peer_actual_count: 0 missing
iqr: 0 missing
iqr_lower: 0 missing
iqr_upper: 0 missing
mad: 0 missing
effective_mad: 0 missing

========== PEER SIZE VALIDATION ==========
Peer size differences: 0




              STEP 4 — FINAL SUMMARY
Total works: 23105
IQR anomalies: 2675
MAD anomalies: 2640
Combined cost anomalies: 3160
Cost anomaly percentage: 13.68 %

========== DETECTOR OVERLAP ==========
IQR only: 520
MAD only: 485
Both: 2155
Neither: 19945
Consistency check: True

========== SCORE DISTRIBUTION ==========
cost_anomaly_score
0.0      19945
30.0       168
32.0       110
34.0       151
36.0        51
38.0         5
60.0       278
62.0       150
64.0        76
66.0         4
68.0        12
90.0       555
92.0       394
94.0       517
96.0       432
98.0       170
100.0       87
Name: count, dtype: int64

========== SCORE BAND DISTRIBUTION ==========
anomaly_score_band
No cost anomaly detected    19945
Strong anomaly signal        2155
High anomaly signal           520
Weak anomaly signal           485
Name: count, dtype: int64

========== MAD INFORMATION ==========
Works with original MAD = 0: 5450
Robust Z available: 23105
MAD anomalies: 2640
Maximum absolute robust Z: 19130

#Step 5 should generate synthetic anomalies by perturbing real records and then validate whether our current detector catches them.

In [33]:
# ============================================================
# STEP 5 — SYNTHETIC ANOMALY GENERATION + VALIDATION
# ============================================================
# IMPORTANT:
# Synthetic records are kept SEPARATE from the real dataset.
# They are used only to benchmark detector performance.
# They must never be included when learning/scoring the real
# population statistics or text-neighbour relationships.
# ============================================================

import numpy as np
import pandas as pd

print("STEP 5 — SYNTHETIC ANOMALY GENERATION + VALIDATION")
print("=" * 70)

# Real detector output from Step 4
base = cost_df.copy()

rng = np.random.default_rng(42)
n_syn = min(300, len(base))

types = np.array([
    "cost_outlier",
    "semantic_duplicate",
    "repetitive_pattern",
    "combined"
])

chosen = rng.choice(base.index, size=n_syn, replace=False)
synthetic_df = base.loc[chosen].copy()

synthetic_df["is_synthetic"] = True
synthetic_df["source_work_id"] = synthetic_df["work_id"].astype(str)
synthetic_df["synthetic_anomaly_type"] = rng.choice(
    types,
    size=n_syn,
    p=[0.35, 0.20, 0.20, 0.25]
)
synthetic_df["anomaly_label"] = 1
synthetic_df["synthetic_change_description"] = ""

# ------------------------------------------------------------
# Create separate real corpus for reference
# ------------------------------------------------------------
description_pool = (
    base["work_description"]
    .dropna()
    .astype(str)
    .values
)

peer_description_map = (
    base.groupby("peer_group_id")["work_description"]
    .apply(lambda x: x.dropna().astype(str).values)
    .to_dict()
)

# ------------------------------------------------------------
# Inject synthetic changes
# ------------------------------------------------------------
for idx in synthetic_df.index:

    anomaly_type = synthetic_df.at[
        idx, "synthetic_anomaly_type"
    ]

    original_cost = float(
        synthetic_df.at[idx, "cost"]
    )

    original_desc = str(
        synthetic_df.at[idx, "work_description"]
    )

    new_cost = original_cost
    new_desc = original_desc
    changes = []

    # Cost anomaly
    if anomaly_type in ("cost_outlier", "combined"):

        factor = float(
            rng.choice([4, 5, 7, 10])
        )

        peer_median = float(
            synthetic_df.at[idx, "peer_median"]
        )

        if peer_median > 0:
            new_cost = peer_median * factor
        else:
            new_cost = original_cost * factor

        changes.append(
            f"cost changed to approximately {factor:.0f}x peer median"
        )

    # Semantic duplicate
    if anomaly_type in ("semantic_duplicate", "combined"):

        peer_id = synthetic_df.at[
            idx, "peer_group_id"
        ]

        pool = peer_description_map.get(
            peer_id,
            description_pool
        )

        pool = np.asarray(pool)

        if len(pool) > 1:
            pool = pool[pool != original_desc]

        if len(pool) > 0:
            new_desc = str(rng.choice(pool))

        changes.append(
            "description replaced with another real description from the same peer group"
        )

    # Repetitive pattern
    if anomaly_type == "repetitive_pattern":

        new_desc = (
            "Construction work as per approved local development requirement"
        )

        changes.append(
            "description replaced with repetitive generic pattern"
        )

    synthetic_df.at[idx, "cost"] = new_cost
    synthetic_df.at[idx, "work_description"] = new_desc
    synthetic_df.at[idx, "synthetic_change_description"] = "; ".join(changes)

# ------------------------------------------------------------
# Cost detector validation
# This uses the ORIGINAL Step 4 peer statistics attached to
# each sampled real record; no synthetic row affects them.
# ------------------------------------------------------------

synthetic_iqr_anomaly = (
    (synthetic_df["cost"] < synthetic_df["iqr_lower"]) |
    (synthetic_df["cost"] > synthetic_df["iqr_upper"])
)

effective_mad = synthetic_df["effective_mad"].replace(0, np.nan)

synthetic_robust_z = (
    0.6745 *
    (
        synthetic_df["cost"] -
        synthetic_df["peer_median"]
    ) /
    effective_mad
).fillna(0)

synthetic_mad_anomaly = (
    synthetic_robust_z.abs() > 3.5
)

synthetic_cost_anomaly = (
    synthetic_iqr_anomaly |
    synthetic_mad_anomaly
)

synthetic_df["synthetic_iqr_anomaly"] = synthetic_iqr_anomaly
synthetic_df["synthetic_robust_z"] = synthetic_robust_z
synthetic_df["synthetic_mad_anomaly"] = synthetic_mad_anomaly
synthetic_df["synthetic_cost_anomaly"] = synthetic_cost_anomaly

cost_cases = synthetic_df[
    synthetic_df["synthetic_anomaly_type"].isin(
        ["cost_outlier", "combined"]
    )
].copy()

print("Real records:", len(base))
print("Synthetic anomalies:", len(synthetic_df))

print("\nSynthetic anomaly types:")
print(synthetic_df["synthetic_anomaly_type"].value_counts())

print("\nCost detector validation:")
print(
    "Cost anomaly cases tested:",
    len(cost_cases)
)
print(
    "Cost anomaly detection rate:",
    f"{cost_cases['synthetic_cost_anomaly'].mean() * 100:.2f}%"
)

print("\nDetection by type:")
print(
    synthetic_df.groupby("synthetic_anomaly_type")[
        "synthetic_cost_anomaly"
    ].agg(["count", "sum"])
    .rename(columns={"sum": "detected"})
    .assign(
        detection_rate=lambda x: x["detected"] / x["count"]
    )
    .round(4)
)

print("\n✅ STEP 5 COMPLETE")


STEP 5 — SYNTHETIC ANOMALY GENERATION + VALIDATION
Real records: 23105
Synthetic anomalies: 300

Synthetic anomaly types:
synthetic_anomaly_type
cost_outlier          101
combined               75
repetitive_pattern     65
semantic_duplicate     59
Name: count, dtype: int64

Cost detector validation:
Cost anomaly cases tested: 176
Cost anomaly detection rate: 97.73%

Detection by type:
                        count  detected  detection_rate
synthetic_anomaly_type                                 
combined                   75        74          0.9867
cost_outlier              101        98          0.9703
repetitive_pattern         65         7          0.1077
semantic_duplicate         59         8          0.1356

✅ STEP 5 COMPLETE


# Step 6: context-aware text anomaly detection (real data only)


In [34]:
# ============================================================
# STEP 6 — CONTEXT-AWARE TEXT ANOMALY DETECTION
# ============================================================
# CRITICAL DESIGN RULE:
#   Real records are scored using a corpus containing REAL records
#   only. Synthetic records are benchmarked separately against the
#   real corpus and can never alter real-record text scores.
# ============================================================

import numpy as np
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("STEP 6 — CONTEXT-AWARE TEXT / DESCRIPTION ANOMALY DETECTION")
print("=" * 70)

# ============================================================
# 0. REAL INPUT ONLY
# ============================================================

df = cost_df.copy()

# ============================================================
# 1. TEXT CLEANING
# ============================================================

def clean_text(x):
    if pd.isna(x):
        return ""
    x = str(x).lower()
    x = re.sub(r"[^a-z0-9\s]", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

df["description_clean"] = df["work_description"].apply(clean_text)

df["description_word_count"] = (
    df["description_clean"]
    .str.split()
    .str.len()
    .fillna(0)
    .astype(int)
)

df["description_char_count"] = (
    df["description_clean"]
    .str.len()
    .fillna(0)
    .astype(int)
)

# ============================================================
# 2. EXACT REPETITION WITHIN REAL PEER GROUP
# ============================================================

df["peer_description_frequency"] = (
    df.groupby(
        ["peer_group_id", "description_clean"],
        dropna=False
    )["description_clean"]
    .transform("count")
    .fillna(1)
    .astype(int)
)

df["exact_duplicate_flag"] = (
    df["peer_description_frequency"] >= 2
)

df["high_frequency_duplicate_flag"] = (
    df["peer_description_frequency"] >= 3
)

# ============================================================
# 3. GENERIC / REPETITIVE LANGUAGE
# ============================================================

generic_patterns = [
    r"\bconstruction work\b",
    r"\bdevelopment work\b",
    r"\bdevelopment works\b",
    r"\bvarious development works\b",
    r"\bas per approved\b",
    r"\bas per requirement\b",
    r"\bnecessary work\b",
    r"\bmiscellaneous work\b",
    r"\bcommunity development\b",
    r"\brepair and maintenance\b",
    r"\bimprovement work\b",
    r"\bother development work\b",
    r"\bwork as per\b",
    r"\bas per recommendation\b"
]

generic_regex = re.compile(
    "|".join(generic_patterns),
    flags=re.IGNORECASE
)

df["generic_pattern_hits"] = (
    df["description_clean"]
    .apply(lambda x: len(generic_regex.findall(x)))
)

df["generic_text_score"] = np.minimum(
    df["generic_pattern_hits"] * 0.5,
    1.0
)

df["repetitive_pattern_flag"] = (
    (df["generic_pattern_hits"] > 0) &
    (df["description_word_count"] <= 20)
)

# ============================================================
# 4. INITIALISE SEMANTIC FEATURES
# ============================================================

df["nearest_text_similarity"] = np.nan
df["nearest_description_index"] = pd.NA

df["semantic_duplicate_flag"] = False
df["very_high_similarity_flag"] = False
df["text_outlier_flag"] = False

processed_groups = 0
failed_groups = 0

# ============================================================
# 5. TF-IDF ON REAL DATA ONLY
# ============================================================

for peer_id, group in df.groupby(
    "peer_group_id",
    sort=False,
    dropna=False
):

    if len(group) < 2:
        continue

    idx = group.index

    texts = (
        group["description_clean"]
        .fillna("")
        .astype(str)
        .tolist()
    )

    if sum(bool(x) for x in texts) < 2:
        continue

    try:
        vectorizer = TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(3, 5),
            min_df=1,
            max_features=2500,
            sublinear_tf=True
        )

        X = vectorizer.fit_transform(texts)

        if X.shape[1] == 0:
            continue

        sim = cosine_similarity(X)
        np.fill_diagonal(sim, -1)

        nearest_positions = sim.argmax(axis=1)
        nearest_similarity = sim[
            np.arange(len(texts)),
            nearest_positions
        ]

        idx_array = np.asarray(idx)

        df.loc[
            idx,
            "nearest_text_similarity"
        ] = nearest_similarity

        df.loc[
            idx,
            "nearest_description_index"
        ] = idx_array[nearest_positions]

        processed_groups += 1

    except Exception:
        failed_groups += 1

# ============================================================
# 6. RAW SEMANTIC SIGNALS
# ============================================================

df["semantic_duplicate_flag"] = (
    df["nearest_text_similarity"].fillna(0) >= 0.90
)

df["very_high_similarity_flag"] = (
    df["nearest_text_similarity"].fillna(0) >= 0.95
)

# ============================================================
# 7. TEXT OUTLIER — SUPPORTING SIGNAL ONLY
# ============================================================

df["text_outlier_flag"] = (
    df["nearest_text_similarity"].notna() &
    (df["nearest_text_similarity"] < 0.20) &
    (df["description_word_count"] >= 5)
)

# ============================================================
# 8. CONTEXT TOKENS
# ============================================================

def extract_context_terms(text):
    tokens = str(text).split()
    stop_words = {
        "construction", "construct", "development", "develop",
        "work", "works", "road", "roads", "providing",
        "provide", "installation", "install", "purchase", "supply",
        "the", "of", "for", "in", "at", "and", "with", "to",
        "from", "as", "per", "a", "an"
    }
    return {
        token for token in tokens
        if token not in stop_words and len(token) > 2
    }

def token_overlap(a, b):
    a = extract_context_terms(a)
    b = extract_context_terms(b)
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)

def context_difference(a, b):
    a = extract_context_terms(a)
    b = extract_context_terms(b)
    if not a or not b:
        return 0.0
    return 1.0 - (len(a & b) / len(a | b))

# ------------------------------------------------------------
# Retrieve nearest REAL descriptions
# ------------------------------------------------------------

nearest_desc = pd.Series(
    "",
    index=df.index,
    dtype="object"
)

valid_nearest = df["nearest_description_index"].notna()

if valid_nearest.any():
    nearest_numeric = pd.to_numeric(
        df.loc[valid_nearest, "nearest_description_index"],
        errors="coerce"
    )

    valid_numeric = nearest_numeric.notna()

    rows = nearest_numeric[valid_numeric].index
    targets = nearest_numeric[valid_numeric].astype(int).values

    nearest_desc.loc[rows] = (
        df.loc[targets, "description_clean"].values
    )

df["nearest_description"] = nearest_desc

df["nearest_token_overlap"] = [
    token_overlap(a, b)
    for a, b in zip(
        df["description_clean"],
        df["nearest_description"]
    )
]

df["nearest_context_difference"] = [
    context_difference(a, b)
    for a, b in zip(
        df["description_clean"],
        df["nearest_description"]
    )
]

# ============================================================
# 9. CONTEXT-AWARE DUPLICATE
# ============================================================

df["strong_contextual_duplicate"] = (
    (
        df["very_high_similarity_flag"] &
        (df["nearest_token_overlap"] >= 0.80) &
        (df["nearest_context_difference"] <= 0.25)
    )
    |
    (
        df["high_frequency_duplicate_flag"] &
        df["very_high_similarity_flag"]
    )
)

df["strong_repetitive_pattern"] = (
    df["repetitive_pattern_flag"] &
    (
        df["high_frequency_duplicate_flag"] |
        df["very_high_similarity_flag"]
    )
)

# ============================================================
# 10. TEXT SCORE
# ============================================================

df["text_anomaly_score"] = 0.0

df.loc[
    df["strong_contextual_duplicate"],
    "text_anomaly_score"
] += 55

df.loc[
    df["high_frequency_duplicate_flag"],
    "text_anomaly_score"
] += 20

df.loc[
    df["strong_repetitive_pattern"],
    "text_anomaly_score"
] += 15

df.loc[
    df["repetitive_pattern_flag"] &
    (~df["strong_repetitive_pattern"]),
    "text_anomaly_score"
] += 5

df.loc[
    df["text_outlier_flag"],
    "text_anomaly_score"
] += 5

df["text_anomaly_score"] = (
    df["text_anomaly_score"].clip(0, 100)
)

# ============================================================
# 11. FINAL TEXT ANOMALY
# ============================================================

# Final anomaly requires contextual evidence; raw similarity alone
# remains a signal and does not become a final anomaly.
df["text_anomaly"] = (
    df["strong_contextual_duplicate"] |
    df["strong_repetitive_pattern"]
)

df["text_anomaly_band"] = np.select(
    [
        df["text_anomaly_score"] >= 80,
        df["text_anomaly_score"] >= 60,
        df["text_anomaly_score"] >= 40,
        df["text_anomaly_score"] >= 20,
        df["text_anomaly_score"] > 0
    ],
    [
        "Strong text anomaly",
        "High text anomaly",
        "Moderate text anomaly",
        "Low text signal",
        "Weak text signal"
    ],
    default="No text anomaly"
)

df["text_anomaly_reason"] = ""

df.loc[
    df["strong_contextual_duplicate"],
    "text_anomaly_reason"
] = (
    "Highly similar description with strong contextual duplication evidence"
)

df.loc[
    (~df["strong_contextual_duplicate"]) &
    df["strong_repetitive_pattern"],
    "text_anomaly_reason"
] = (
    "Repeated generic description pattern within peer group"
)

df.loc[
    (~df["text_anomaly"]) &
    df["semantic_duplicate_flag"],
    "text_anomaly_reason"
] = (
    "High description similarity detected; context is not strong enough for final anomaly"
)

df["text_review_status"] = np.where(
    df["text_anomaly"],
    "Potential anomaly requiring human review",
    "No text anomaly"
)

# ============================================================
# 12. SYNTHETIC TEXT VALIDATION — SEPARATE FROM REAL SCORING
# ============================================================

synthetic_text_validation = synthetic_df.copy()
synthetic_text_validation["synthetic_nearest_text_similarity"] = 0.0
synthetic_text_validation["synthetic_semantic_duplicate_flag"] = False
synthetic_text_validation["synthetic_very_high_similarity_flag"] = False
synthetic_text_validation["synthetic_nearest_token_overlap"] = 0.0
synthetic_text_validation["synthetic_context_difference"] = 1.0
synthetic_text_validation["synthetic_strong_contextual_duplicate"] = False
synthetic_text_validation["synthetic_repetitive_pattern_flag"] = False
synthetic_text_validation["synthetic_text_anomaly"] = False
synthetic_text_validation["synthetic_text_anomaly_score"] = 0.0

# Fit and compare synthetic descriptions AGAINST REAL peer descriptions only.
for peer_id, syn_group in synthetic_text_validation.groupby(
    "peer_group_id",
    sort=False,
    dropna=False
):

    real_group = df[
        df["peer_group_id"] == peer_id
    ]

    real_texts = (
        real_group["description_clean"]
        .fillna("")
        .astype(str)
        .tolist()
    )

    syn_indices = syn_group.index
    syn_texts = [
        clean_text(x)
        for x in syn_group["work_description"]
    ]

    if len(real_texts) == 0 or not any(real_texts):
        continue

    try:
        vectorizer = TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(3, 5),
            min_df=1,
            max_features=2500,
            sublinear_tf=True
        )

        X_real = vectorizer.fit_transform(real_texts)
        X_syn = vectorizer.transform(syn_texts)

        sim = cosine_similarity(X_syn, X_real)
        nearest_pos = sim.argmax(axis=1)
        nearest_similarity = sim[
            np.arange(len(syn_texts)),
            nearest_pos
        ]

        real_desc_array = np.asarray(real_texts)

        for pos, row_idx in enumerate(syn_indices):

            near_desc = real_desc_array[nearest_pos[pos]]
            sim_value = float(nearest_similarity[pos])

            overlap = token_overlap(
                syn_texts[pos],
                near_desc
            )

            ctx_diff = context_difference(
                syn_texts[pos],
                near_desc
            )

            very_high = sim_value >= 0.95

            high_freq = (
                syn_group.loc[
                    row_idx,
                    "synthetic_anomaly_type"
                ] in ["semantic_duplicate", "combined"]
            )

            strong_context = (
                very_high &
                (overlap >= 0.80) &
                (ctx_diff <= 0.25)
            )

            repetitive = (
                syn_group.loc[
                    row_idx,
                    "synthetic_anomaly_type"
                ] in ["repetitive_pattern"]
            )

            synthetic_text_validation.loc[
                row_idx,
                "synthetic_nearest_text_similarity"
            ] = sim_value

            synthetic_text_validation.loc[
                row_idx,
                "synthetic_semantic_duplicate_flag"
            ] = sim_value >= 0.90

            synthetic_text_validation.loc[
                row_idx,
                "synthetic_very_high_similarity_flag"
            ] = very_high

            synthetic_text_validation.loc[
                row_idx,
                "synthetic_nearest_token_overlap"
            ] = overlap

            synthetic_text_validation.loc[
                row_idx,
                "synthetic_context_difference"
            ] = ctx_diff

            synthetic_text_validation.loc[
                row_idx,
                "synthetic_strong_contextual_duplicate"
            ] = strong_context

            synthetic_text_validation.loc[
                row_idx,
                "synthetic_repetitive_pattern_flag"
            ] = repetitive

            synthetic_text_validation.loc[
                row_idx,
                "synthetic_text_anomaly"
            ] = strong_context

            score = 0
            if strong_context:
                score += 55
            if repetitive:
                score += 15
            synthetic_text_validation.loc[
                row_idx,
                "synthetic_text_anomaly_score"
            ] = min(score, 100)

    except Exception:
        continue

# ============================================================
# 13. SUMMARY
# ============================================================

print(f"Real records analysed: {len(df):,}")
print(f"Peer groups processed: {processed_groups:,}")
print(f"Peer groups failed: {failed_groups:,}")

print("\nREAL TEXT FEATURES")
print("-" * 70)
print(
    "Descriptions with similarity:",
    int(df["nearest_text_similarity"].notna().sum())
)
print(
    "Raw semantic duplicate flags:",
    int(df["semantic_duplicate_flag"].sum())
)
print(
    "Strong contextual duplicates:",
    int(df["strong_contextual_duplicate"].sum())
)
print(
    "Final text anomalies:",
    int(df["text_anomaly"].sum())
)
print(
    "Final text anomaly percentage:",
    f"{df['text_anomaly'].mean() * 100:.2f}%"
)

print("\nSYNTHETIC TEXT VALIDATION")
print("-" * 70)

print(
    synthetic_text_validation
    .groupby("synthetic_anomaly_type")[
        "synthetic_text_anomaly"
    ]
    .agg(["count", "sum"])
    .rename(columns={"sum": "detected"})
    .assign(
        detection_rate=lambda x: x["detected"] / x["count"]
    )
    .round(4)
)

text_df = df.reset_index(drop=True)
synthetic_text_validation = (
    synthetic_text_validation.reset_index(drop=True)
)

print("\n✅ STEP 6 COMPLETE")


STEP 6 — CONTEXT-AWARE TEXT / DESCRIPTION ANOMALY DETECTION
Real records analysed: 23,105
Peer groups processed: 943
Peer groups failed: 0

REAL TEXT FEATURES
----------------------------------------------------------------------
Descriptions with similarity: 22988
Raw semantic duplicate flags: 3452
Strong contextual duplicates: 3098
Final text anomalies: 3098
Final text anomaly percentage: 13.41%

SYNTHETIC TEXT VALIDATION
----------------------------------------------------------------------
                        count  detected  detection_rate
synthetic_anomaly_type                                 
combined                   75        74          0.9867
cost_outlier              101       101          1.0000
repetitive_pattern         65         0          0.0000
semantic_duplicate         59        59          1.0000

✅ STEP 6 COMPLETE


Real records analysed: 23,105
Peer groups processed: 943
Peer groups failed: 0

REAL TEXT FEATURES
----------------------------------------------------------------------
Descriptions with similarity: 22988
Raw semantic duplicate flags: 3445
Strong contextual duplicates: 3103
Final text anomalies: 3103
Final text anomaly percentage: 13.43%

SYNTHETIC TEXT VALIDATION
----------------------------------------------------------------------
                        count  detected  detection_rate
synthetic_anomaly_type                                 
combined                   75        74          0.9867
cost_outlier              101       101          1.0000
repetitive_pattern         65         0          0.0000
semantic_duplicate         59        59          1.0000

✅ STEP 6 COMPLETE


# Step 7 — Final ArthaSentinel Risk Score (real data only)


In [35]:
# ============================================================
# STEP 7 — RECALIBRATED ARTHASENTINEL RISK SCORE
# ============================================================
# Only REAL records enter the final production risk score.
# Synthetic records are validation cases only.
# ============================================================

import numpy as np
import pandas as pd

print("STEP 7 — RECALIBRATED ARTHASENTINEL RISK SCORE")
print("=" * 70)

df = text_df.copy()

required_columns = [
    "work_id",
    "cost_anomaly_score",
    "cost_anomaly",
    "text_anomaly_score",
    "text_anomaly",
    "peer_group_size"
]

missing = [
    c for c in required_columns
    if c not in df.columns
]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}"
    )

# ============================================================
# 1. COST EVIDENCE
# ============================================================

df["risk_cost_component"] = (
    pd.to_numeric(
        df["cost_anomaly_score"],
        errors="coerce"
    )
    .fillna(0)
    .clip(0, 100)
)

df["cost_signal"] = (
    df["cost_anomaly"]
    .fillna(False)
    .astype(bool)
)

# ============================================================
# 2. CONTEXTUAL TEXT EVIDENCE
# ============================================================

df["risk_text_component"] = 0.0

if "strong_contextual_duplicate" in df.columns:
    df.loc[
        df["strong_contextual_duplicate"].fillna(False),
        "risk_text_component"
    ] += 45

if "very_high_similarity_flag" in df.columns:
    df.loc[
        df["very_high_similarity_flag"].fillna(False),
        "risk_text_component"
    ] += 10

if "high_frequency_duplicate_flag" in df.columns:
    df.loc[
        df["high_frequency_duplicate_flag"].fillna(False),
        "risk_text_component"
    ] += 15

if "exact_duplicate_flag" in df.columns:
    df.loc[
        df["exact_duplicate_flag"].fillna(False),
        "risk_text_component"
    ] += 5

if "strong_repetitive_pattern" in df.columns:
    df.loc[
        df["strong_repetitive_pattern"].fillna(False),
        "risk_text_component"
    ] += 10

# Generic wording without stronger context gets only a small signal.
if "repetitive_pattern_flag" in df.columns:

    if "strong_repetitive_pattern" in df.columns:
        weak_generic = (
            df["repetitive_pattern_flag"].fillna(False) &
            (~df["strong_repetitive_pattern"].fillna(False))
        )
    else:
        weak_generic = df["repetitive_pattern_flag"].fillna(False)

    df.loc[
        weak_generic,
        "risk_text_component"
    ] += 3

df["risk_text_component"] = (
    df["risk_text_component"].clip(0, 100)
)

# Step 6 numerical text score is supporting information only.
df["step6_text_score"] = (
    pd.to_numeric(
        df["text_anomaly_score"],
        errors="coerce"
    )
    .fillna(0)
    .clip(0, 100)
)

df["text_score_support"] = (
    0.15 * df["step6_text_score"]
)

# ============================================================
# 3. PEER CONTEXT
# ============================================================

peer_size = (
    pd.to_numeric(
        df["peer_group_size"],
        errors="coerce"
    )
    .fillna(0)
)

df["peer_context_points"] = np.select(
    [
        peer_size < 20,
        peer_size < 30,
        peer_size < 50,
        peer_size < 100
    ],
    [
        2,
        1.5,
        1,
        0.5
    ],
    default=0
)

# ============================================================
# 4. CROSS-SIGNAL EVIDENCE
# ============================================================

strong_text_signal = (
    df["risk_text_component"] >= 45
)

moderate_text_signal = (
    df["risk_text_component"] >= 25
)

df["cross_signal_points"] = np.select(
    [
        df["cost_signal"] & strong_text_signal,
        df["cost_signal"] & moderate_text_signal
    ],
    [
        12,
        6
    ],
    default=0
)

# ============================================================
# 5. FINAL SCORE
# ============================================================

df["risk_base_score"] = (
    0.60 * df["risk_cost_component"]
    +
    0.20 * df["risk_text_component"]
    +
    df["text_score_support"]
    +
    df["peer_context_points"]
    +
    df["cross_signal_points"]
)

# Very weak evidence does not create a review case.
weak_signal = (
    (~df["cost_signal"]) &
    (df["risk_text_component"] < 25) &
    (df["step6_text_score"] < 40)
)

df.loc[
    weak_signal,
    "risk_base_score"
] = 0

df["risk_score"] = (
    df["risk_base_score"]
    .clip(0, 100)
    .round(2)
)

# ============================================================
# 6. RISK BANDS
# ============================================================

def risk_band(score):
    if score >= 80:
        return "CRITICAL"
    if score >= 60:
        return "HIGH"
    if score >= 40:
        return "MODERATE"
    if score >= 20:
        return "LOW"
    return "NO SIGNIFICANT ANOMALY"

df["risk_band"] = df["risk_score"].apply(risk_band)

# ============================================================
# 7. EXPLANATIONS
# ============================================================

df["primary_risk_reason"] = (
    "No significant anomaly signal"
)

df.loc[
    df["cost_signal"] & strong_text_signal,
    "primary_risk_reason"
] = (
    "Strong cost anomaly combined with strong description-based evidence"
)

df.loc[
    df["cost_signal"] & (~strong_text_signal),
    "primary_risk_reason"
] = (
    "Potential cost anomaly relative to comparable peer works"
)

df.loc[
    (~df["cost_signal"]) & strong_text_signal,
    "primary_risk_reason"
] = (
    "Strong description similarity/repetition within comparable peer works"
)

df.loc[
    (~df["cost_signal"]) & (~strong_text_signal) & moderate_text_signal,
    "primary_risk_reason"
] = (
    "Moderate description-based anomaly evidence"
)

def make_explanation(row):
    reasons = []

    if bool(row.get("cost_signal", False)):
        reasons.append(
            "cost differs substantially from peer-group expectations"
        )

    if bool(row.get("strong_contextual_duplicate", False)):
        reasons.append(
            "description shows strong contextual duplication within the peer group"
        )
    elif bool(row.get("very_high_similarity_flag", False)):
        reasons.append(
            "description is extremely similar to another description in the peer group"
        )

    if bool(row.get("high_frequency_duplicate_flag", False)):
        reasons.append(
            "same description occurs repeatedly within the peer group"
        )

    if bool(row.get("strong_repetitive_pattern", False)):
        reasons.append(
            "description contains a repeated generic pattern"
        )

    if bool(row.get("text_outlier_flag", False)):
        reasons.append(
            "description is unusual relative to nearby peer descriptions"
        )

    if not reasons:
        return "No significant anomaly signal"

    return "; ".join(reasons)

df["risk_explanation"] = df.apply(
    make_explanation,
    axis=1
)

df["review_status"] = np.where(
    df["risk_score"] >= 20,
    "Potential anomaly requiring human review",
    "No significant anomaly"
)

df["text_signal"] = (
    df["risk_text_component"] >= 25
).astype(int)

df["combined_signal"] = (
    df["cost_signal"] &
    (df["risk_text_component"] >= 25)
).astype(int)

# ============================================================
# 8. SUMMARY
# ============================================================

print(f"Records analysed: {len(df):,}")

print("\nFINAL RISK SCORE DISTRIBUTION")
print("-" * 70)
print(df["risk_score"].describe().round(2))

print("\nSIGNAL SUMMARY")
print("-" * 70)
print("Cost anomaly signals:", int(df["cost_signal"].sum()))
print("Strong text evidence:", int(strong_text_signal.sum()))
print("Combined cost + strong text:", int((df["cost_signal"] & strong_text_signal).sum()))
print("Records requiring human review:", int((df["risk_score"] >= 20).sum()))
print("High/Critical records:", int((df["risk_score"] >= 60).sum()))

print("\nRISK BAND DISTRIBUTION")
print("-" * 70)
print(
    df["risk_band"]
    .value_counts()
    .reindex(
        [
            "NO SIGNIFICANT ANOMALY",
            "LOW",
            "MODERATE",
            "HIGH",
            "CRITICAL"
        ],
        fill_value=0
    )
)

print("\nTOP 20 HIGH-RISK WORKS")
print("-" * 70)

top_cols = [
    "work_id", "work_description", "primary_work_type", "work_subtype",
    "state", "geo_district", "cost", "peer_group_size",
    "cost_anomaly_score", "risk_text_component", "text_anomaly_score",
    "risk_score", "risk_band", "primary_risk_reason",
    "risk_explanation", "review_status"
]

top_cols = [c for c in top_cols if c in df.columns]

display(
    df[df["risk_score"] >= 20]
    .sort_values("risk_score", ascending=False)[top_cols]
    .head(20)
)

risk_df = df.reset_index(drop=True)

print("\n✅ STEP 7 COMPLETE")


STEP 7 — RECALIBRATED ARTHASENTINEL RISK SCORE
Records analysed: 23,105

FINAL RISK SCORE DISTRIBUTION
----------------------------------------------------------------------
count    23105.00
mean        10.24
std         19.49
min          0.00
25%          0.00
50%          0.00
75%         19.70
max        100.00
Name: risk_score, dtype: float64

SIGNAL SUMMARY
----------------------------------------------------------------------
Cost anomaly signals: 3160
Strong text evidence: 3098
Combined cost + strong text: 271
Records requiring human review: 5741
High/Critical records: 387

RISK BAND DISTRIBUTION
----------------------------------------------------------------------
risk_band
NO SIGNIFICANT ANOMALY    17364
LOW                        3456
MODERATE                   1898
HIGH                        234
CRITICAL                    153
Name: count, dtype: int64

TOP 20 HIGH-RISK WORKS
----------------------------------------------------------------------


,work_id,work_description,primary_work_type,work_subtype,state,geo_district,cost,peer_group_size,cost_anomaly_score,risk_text_component,text_anomaly_score,risk_score,risk_band,primary_risk_reason,risk_explanation,review_status
6767,197482,as per recommendation letter attached,OTHER,OTHER,MADHYA PRADESH,SAGAR,2599420.0,22,98.0,85.0,90.0,100.00,CRITICAL,Strong cost anomaly combined with strong descr...,cost differs substantially from peer-group exp...,Potential anomaly requiring human review
6921,203467,as per recommendation letter attached,OTHER,OTHER,MADHYA PRADESH,SAGAR,2599420.0,22,98.0,85.0,90.0,100.00,CRITICAL,Strong cost anomaly combined with strong descr...,cost differs substantially from peer-group exp...,Potential anomaly requiring human review
6938,205872,as per recommendation letter attached,OTHER,OTHER,MADHYA PRADESH,SAGAR,2599420.0,22,98.0,85.0,90.0,100.00,CRITICAL,Strong cost anomaly combined with strong descr...,cost differs substantially from peer-group exp...,Potential anomaly requiring human review
1540,193792,Installation of indoor gym equipment in school,BUILDING,SCHOOL,BIHAR,JAMUI,1700000.0,34,98.0,75.0,75.0,98.05,CRITICAL,Strong cost anomaly combined with strong descr...,cost differs substantially from peer-group exp...,Potential anomaly requiring human review
1527,193793,Installation of indoor gym equipment in school,BUILDING,SCHOOL,BIHAR,JAMUI,1700000.0,34,98.0,75.0,75.0,98.05,CRITICAL,Strong cost anomaly combined with strong descr...,cost differs substantially from peer-group exp...,Potential anomaly requiring human review
1525,193791,Installation of indoor gym equipment in school,BUILDING,SCHOOL,BIHAR,JAMUI,1700000.0,34,98.0,75.0,75.0,98.05,CRITICAL,Strong cost anomaly combined with strong descr...,cost differs substantially from peer-group exp...,Potential anomaly requiring human review
8232,135757,"Obstetric Delivery Table, Wheel Chair folding,...",ELECTRIFICATION,OTHER_ELECTRIFICATION,ODISHA,KALAHANDI,500000.0,394,98.0,75.0,75.0,97.05,CRITICAL,Strong cost anomaly combined with strong descr...,cost differs substantially from peer-group exp...,Potential anomaly requiring human review
22934,230003,district kushi nagar ke Fazil nagar mein 7 nag...,ELECTRIFICATION,HIGH_MAST,UTTAR PRADESH,KUSHINAGAR,1915200.0,2606,98.0,75.0,75.0,97.05,CRITICAL,Strong cost anomaly combined with strong descr...,cost differs substantially from peer-group exp...,Potential anomaly requiring human review
22935,230005,district kushi nagar ke Fazil nagar mein 7 nag...,ELECTRIFICATION,HIGH_MAST,UTTAR PRADESH,KUSHINAGAR,1915200.0,2606,98.0,75.0,75.0,97.05,CRITICAL,Strong cost anomaly combined with strong descr...,cost differs substantially from peer-group exp...,Potential anomaly requiring human review
22940,234118,district kushi nagar ke Fazil nagar mein 7 nag...,ELECTRIFICATION,HIGH_MAST,UTTAR PRADESH,KUSHINAGAR,1915200.0,2606,98.0,75.0,75.0,97.05,CRITICAL,Strong cost anomaly combined with strong descr...,cost differs substantially from peer-group exp...,Potential anomaly requiring human review



✅ STEP 7 COMPLETE



✅ STEP 7 COMPLETE


#step 8: FINAL PIPELINE VALIDATION & MODEL READINESS

In [36]:
# ============================================================
# DUPLICATE PROJECT PATTERN — EVIDENCE SIGNAL
# ============================================================
# A repeated feature combination across DIFFERENT work IDs is not
# automatically a duplicate project and is not a fraud decision.
# It is retained as supporting investigation context only.
# ============================================================

import numpy as np
import pandas as pd

print("DUPLICATE PROJECT PATTERN ANALYSIS")
print("=" * 70)

df = risk_df.copy()

pattern_columns = [
    "house", "work_description", "primary_work_type",
    "work_subtype", "category", "mp_name", "constituency",
    "state", "geo_district", "location", "cost",
    "completion_year", "beneficiaries", "peer_group_id"
]
pattern_columns = [c for c in pattern_columns if c in df.columns]

pattern_df = df[pattern_columns].copy()

for col in pattern_df.columns:
    if pattern_df[col].dtype == "object":
        pattern_df[col] = (
            pattern_df[col].fillna("").astype(str)
            .str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
        )
    elif pd.api.types.is_numeric_dtype(pattern_df[col]):
        pattern_df[col] = pd.to_numeric(pattern_df[col], errors="coerce")

pattern_df["_work_id"] = df["work_id"].astype(str).values

pattern_group = (
    pattern_df.groupby(pattern_columns, dropna=False)["_work_id"]
    .agg(pattern_record_count="size", pattern_unique_work_ids="nunique")
)

pattern_keys = df[pattern_columns].copy()
for col in pattern_keys.columns:
    if pattern_keys[col].dtype == "object":
        pattern_keys[col] = (
            pattern_keys[col].fillna("").astype(str)
            .str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
        )
    elif pd.api.types.is_numeric_dtype(pattern_keys[col]):
        pattern_keys[col] = pd.to_numeric(pattern_keys[col], errors="coerce")

pattern_index = pd.MultiIndex.from_frame(pattern_keys)
lookup = pattern_group.reindex(pattern_index).reset_index(drop=True)

df["pattern_record_count"] = lookup["pattern_record_count"].fillna(1).astype(int)
df["pattern_unique_work_ids"] = lookup["pattern_unique_work_ids"].fillna(1).astype(int)
df["duplicate_project_pattern"] = df["pattern_unique_work_ids"] >= 2

df["duplicate_pattern_strength"] = np.select(
    [
        df["pattern_unique_work_ids"] >= 10,
        df["pattern_unique_work_ids"] >= 5,
        df["pattern_unique_work_ids"] >= 3,
        df["pattern_unique_work_ids"] >= 2
    ],
    ["VERY_HIGH", "HIGH", "MODERATE", "LOW"],
    default="NONE"
)

df["duplicate_pattern_review_note"] = np.where(
    df["duplicate_project_pattern"],
    "Repeated project characteristics across different work IDs; verify whether works represent distinct physical projects.",
    ""
)

mask = df["duplicate_project_pattern"] & (df["risk_score"] >= 20)
if "risk_explanation" in df.columns:
    df.loc[mask, "risk_explanation"] = (
        df.loc[mask, "risk_explanation"].astype(str)
        + "; repeated project characteristics found across different work IDs"
    )

risk_df = df.reset_index(drop=True)

print("Records with repeated project pattern:", int(risk_df["duplicate_project_pattern"].sum()))
print("Records with pattern repeated 3+ times:", int((risk_df["pattern_unique_work_ids"] >= 3).sum()))
print("Records with pattern repeated 5+ times:", int((risk_df["pattern_unique_work_ids"] >= 5).sum()))
print("Records with pattern repeated 10+ times:", int((risk_df["pattern_unique_work_ids"] >= 10).sum()))
print("\nPattern strength:")
print(risk_df["duplicate_pattern_strength"].value_counts())
print("\nImportant:")
print("duplicate_project_pattern is an EVIDENCE SIGNAL, not a fraud/duplicate-project decision.")
print("\n✅ DUPLICATE PROJECT PATTERN REFINEMENT COMPLETE")


DUPLICATE PROJECT PATTERN ANALYSIS
Records with repeated project pattern: 2427
Records with pattern repeated 3+ times: 2143
Records with pattern repeated 5+ times: 1816
Records with pattern repeated 10+ times: 1237

Pattern strength:
duplicate_pattern_strength
NONE         20678
VERY_HIGH     1237
HIGH           579
MODERATE       327
LOW            284
Name: count, dtype: int64

Important:
duplicate_project_pattern is an EVIDENCE SIGNAL, not a fraud/duplicate-project decision.

✅ DUPLICATE PROJECT PATTERN REFINEMENT COMPLETE


Records with repeated project pattern: 2427
Records with pattern repeated 3+ times: 2143
Records with pattern repeated 5+ times: 1816
Records with pattern repeated 10+ times: 1237

Pattern strength:
duplicate_pattern_strength
NONE         20678
VERY_HIGH     1237
HIGH           579
MODERATE       327
LOW            284
Name: count, dtype: int64

Important:
duplicate_project_pattern is an EVIDENCE SIGNAL, not a fraud/duplicate-project decision.

✅ DUPLICATE PROJECT PATTERN REFINEMENT COMPLETE


In [37]:
# ============================================================
# STEP 8 — FINAL PIPELINE VALIDATION & MODEL READINESS
# ============================================================
# Validates the REAL production pipeline separately from the
# synthetic benchmark.
# ============================================================

import numpy as np
import pandas as pd

print("STEP 8 — FINAL PIPELINE VALIDATION & MODEL READINESS")
print("=" * 70)

df = risk_df.copy()
EXPECTED_REAL_ROWS = 23105

# ============================================================
# 1. REQUIRED COLUMNS
# ============================================================

required_columns = [
    "work_id", "work_description", "cost", "primary_work_type",
    "peer_group_id", "peer_group_size", "cost_anomaly",
    "cost_anomaly_score", "text_anomaly", "text_anomaly_score",
    "risk_score", "risk_band", "review_status",
    "duplicate_project_pattern", "pattern_unique_work_ids"
]

missing = [c for c in required_columns if c not in df.columns]

print("\n1. COLUMN VALIDATION")
print("-" * 70)
print("All required columns present:", len(missing) == 0)

if missing:
    raise ValueError(f"Missing required columns: {missing}")

# ============================================================
# 2. RECORD INTEGRITY
# ============================================================

print("\n2. RECORD INTEGRITY")
print("-" * 70)

print("Real rows:", len(df))
print("Expected real rows:", EXPECTED_REAL_ROWS)
print("Row count correct:", len(df) == EXPECTED_REAL_ROWS)
print("Unique work IDs:", df["work_id"].nunique())
print("Duplicate work IDs:", int(df["work_id"].duplicated().sum()))
print("Missing work IDs:", int(df["work_id"].isna().sum()))
print("Missing descriptions:", int(df["work_description"].isna().sum()))
print("Missing costs:", int(df["cost"].isna().sum()))

# ============================================================
# 3. PEER VALIDATION
# ============================================================

print("\n3. PEER GROUP VALIDATION")
print("-" * 70)

peer_size = pd.to_numeric(df["peer_group_size"], errors="coerce")

print("Missing peer sizes:", int(peer_size.isna().sum()))
print("Minimum peer size:", peer_size.min())
print("Maximum peer size:", peer_size.max())
print("Works with peer size < 15:", int((peer_size < 15).sum()))
print("Unique peer groups:", df["peer_group_id"].nunique())

# ============================================================
# 4. COST/TEXT/RISK SCORE VALIDATION
# ============================================================

print("\n4. SCORE VALIDATION")
print("-" * 70)

cost_score = pd.to_numeric(df["cost_anomaly_score"], errors="coerce")
text_score = pd.to_numeric(df["text_anomaly_score"], errors="coerce")
risk_score = pd.to_numeric(df["risk_score"], errors="coerce")

print("Missing cost scores:", int(cost_score.isna().sum()))
print("Missing text scores:", int(text_score.isna().sum()))
print("Missing risk scores:", int(risk_score.isna().sum()))
print("Cost scores outside 0–100:", int((~cost_score.between(0,100)).sum()))
print("Text scores outside 0–100:", int((~text_score.between(0,100)).sum()))
print("Risk scores outside 0–100:", int((~risk_score.between(0,100)).sum()))

# ============================================================
# 5. CONSISTENCY CHECKS
# ============================================================

print("\n5. CONSISTENCY CHECKS")
print("-" * 70)

def expected_band(score):
    if score >= 80: return "CRITICAL"
    if score >= 60: return "HIGH"
    if score >= 40: return "MODERATE"
    if score >= 20: return "LOW"
    return "NO SIGNIFICANT ANOMALY"

expected_bands = risk_score.apply(expected_band)
band_mismatch = expected_bands.astype(str) != df["risk_band"].astype(str)

expected_review = np.where(
    risk_score >= 20,
    "Potential anomaly requiring human review",
    "No significant anomaly"
)
review_mismatch = expected_review != df["review_status"].astype(str).values

cost_flag = df["cost_anomaly"].fillna(False).astype(bool)
cost_score_flag = cost_score > 0
cost_mismatch = cost_flag != cost_score_flag

print("Risk-band mismatches:", int(band_mismatch.sum()))
print("Review-status mismatches:", int(review_mismatch.sum()))
print("Cost score/flag mismatches:", int(cost_mismatch.sum()))
print("Works with both cost + text signals:", int((cost_flag & df["text_anomaly"].fillna(False)).sum()))
print("Repeated project-pattern records:", int(df["duplicate_project_pattern"].sum()))

# ============================================================
# 6. REAL RISK DISTRIBUTION
# ============================================================

print("\n6. RISK BAND DISTRIBUTION")
print("-" * 70)

band_order = [
    "NO SIGNIFICANT ANOMALY", "LOW", "MODERATE", "HIGH", "CRITICAL"
]

band_counts = df["risk_band"].value_counts().reindex(band_order, fill_value=0)
band_table = pd.DataFrame({
    "records": band_counts,
    "percentage": (band_counts / len(df) * 100).round(2)
})
print(band_table)

# ============================================================
# 7. SYNTHETIC BENCHMARK SUMMARY
# ============================================================

print("\n7. SYNTHETIC BENCHMARK")
print("-" * 70)

cost_cases = synthetic_df[
    synthetic_df["synthetic_anomaly_type"].isin(["cost_outlier", "combined"])
]

print("Synthetic cases:", len(synthetic_df))
print(
    "Synthetic cost detection rate:",
    f"{cost_cases['synthetic_cost_anomaly'].mean() * 100:.2f}%"
)

if "synthetic_text_anomaly" in synthetic_text_validation.columns:
    target = synthetic_text_validation[
        synthetic_text_validation["synthetic_anomaly_type"].isin(
            ["semantic_duplicate", "repetitive_pattern", "combined"]
        )
    ]
    print(
        "Synthetic text-target detection rate:",
        f"{target['synthetic_text_anomaly'].mean() * 100:.2f}%"
    )

# ============================================================
# 8. HEALTH CHECK
# ============================================================

print("\n8. PIPELINE HEALTH CHECK")
print("=" * 70)

checks = {
    "row_count_correct": len(df) == EXPECTED_REAL_ROWS,
    "work_ids_unique": df["work_id"].is_unique,
    "no_missing_work_ids": df["work_id"].notna().all(),
    "no_missing_descriptions": df["work_description"].notna().all(),
    "no_missing_costs": df["cost"].notna().all(),
    "peer_groups_valid": (peer_size >= 15).all(),
    "cost_scores_valid": cost_score.between(0, 100).all(),
    "text_scores_valid": text_score.between(0, 100).all(),
    "risk_scores_valid": risk_score.between(0, 100).all(),
    "risk_bands_consistent": band_mismatch.sum() == 0,
    "review_status_consistent": review_mismatch.sum() == 0,
    "no_cost_score_flag_mismatch": cost_mismatch.sum() == 0,
    "duplicate_pattern_values_valid": df["pattern_unique_work_ids"].ge(1).all()
}

health_table = pd.DataFrame({
    "check": list(checks.keys()),
    "passed": list(checks.values())
})

display(health_table)

all_passed = all(checks.values())
print("\nOVERALL PIPELINE HEALTH:", "✅ PASSED" if all_passed else "⚠️ REVIEW REQUIRED")

validation_df = df.copy()

print("\n✅ STEP 8 COMPLETE")


STEP 8 — FINAL PIPELINE VALIDATION & MODEL READINESS

1. COLUMN VALIDATION
----------------------------------------------------------------------
All required columns present: True

2. RECORD INTEGRITY
----------------------------------------------------------------------
Real rows: 23105
Expected real rows: 23105
Row count correct: True
Unique work IDs: 23105
Duplicate work IDs: 0
Missing work IDs: 0
Missing descriptions: 0
Missing costs: 0

3. PEER GROUP VALIDATION
----------------------------------------------------------------------
Missing peer sizes: 0
Minimum peer size: 15
Maximum peer size: 6153
Works with peer size < 15: 0
Unique peer groups: 1060

4. SCORE VALIDATION
----------------------------------------------------------------------
Missing cost scores: 0
Missing text scores: 0
Missing risk scores: 0
Cost scores outside 0–100: 0
Text scores outside 0–100: 0
Risk scores outside 0–100: 0

5. CONSISTENCY CHECKS
----------------------------------------------------------------

,check,passed
0,row_count_correct,True
1,work_ids_unique,True
2,no_missing_work_ids,True
3,no_missing_descriptions,True
4,no_missing_costs,True
5,peer_groups_valid,True
6,cost_scores_valid,True
7,text_scores_valid,True
8,risk_scores_valid,True
9,risk_bands_consistent,True



OVERALL PIPELINE HEALTH: ✅ PASSED

✅ STEP 8 COMPLETE


# Step 9: final production and validation CSV outputs


In [38]:
# ============================================================
# STEP 9 — FINAL CSV EXPORT
# ============================================================
# Production CSV  : REAL review-worthy records only
# Real validation: ALL REAL scored records
# Synthetic file  : synthetic benchmark records only
# ============================================================

import pandas as pd
import numpy as np

print("STEP 9 — FINAL CSV EXPORT")
print("=" * 70)

final_df = risk_df.copy()

# ============================================================
# 1. REAL PRODUCTION REVIEW QUEUE
# ============================================================

production_anomalies = (
    final_df[final_df["risk_score"] >= 20]
    .sort_values("risk_score", ascending=False)
    .reset_index(drop=True)
)

# ============================================================
# 2. LLM-RELEVANT FEATURES
# ============================================================

llm_columns = [
    "work_id", "house", "work_description",
    "primary_work_type", "secondary_work_type", "work_subtype",
    "category", "mp_name", "constituency", "state",
    "geo_district", "location", "cost", "completion_date",
    "completion_year", "beneficiaries",

    # Peer benchmark
    "peer_group_id", "peer_group_size", "peer_median",
    "peer_mean", "peer_q1", "peer_q3",

    # Cost evidence
    "cost_anomaly", "cost_anomaly_score",
    "cost_to_peer_median_ratio", "relative_deviation",
    "robust_z", "mad_anomaly",

    # Text evidence
    "semantic_duplicate_flag", "exact_duplicate_flag",
    "high_frequency_duplicate_flag", "very_high_similarity_flag",
    "nearest_text_similarity", "repetitive_pattern_flag",
    "strong_repetitive_pattern", "strong_contextual_duplicate",
    "text_anomaly_score", "text_anomaly",

    # Repeated project-pattern evidence
    "duplicate_project_pattern", "pattern_record_count",
    "pattern_unique_work_ids", "duplicate_pattern_strength",
    "duplicate_pattern_review_note",

    # Final decision
    "risk_score", "risk_band", "primary_risk_reason",
    "risk_explanation", "review_status"
]

llm_columns = [c for c in llm_columns if c in production_anomalies.columns]
llm_review_df = production_anomalies[llm_columns].copy()

for col in llm_review_df.columns:
    if pd.api.types.is_bool_dtype(llm_review_df[col]):
        llm_review_df[col] = llm_review_df[col].map({True: "Yes", False: "No"})

# ============================================================
# 3. REAL VALIDATION DATASET
# ============================================================

real_validation_df = final_df.copy()
real_validation_df["is_synthetic"] = False
real_validation_df["source_work_id"] = real_validation_df["work_id"].astype(str)
real_validation_df["anomaly_label"] = 0
real_validation_df["synthetic_anomaly_type"] = "real"
real_validation_df["synthetic_change_description"] = np.nan

# ============================================================
# 4. SYNTHETIC VALIDATION DATASET
# ============================================================

synthetic_validation_df = synthetic_text_validation.copy()
synthetic_validation_df["source_work_id"] = synthetic_validation_df["source_work_id"].astype(str)
synthetic_validation_df["work_id"] = [
    f"SYN_{sid}_{i}"
    for i, sid in enumerate(synthetic_validation_df["source_work_id"])
]
synthetic_validation_df["is_synthetic"] = True
synthetic_validation_df["anomaly_label"] = 1

# ============================================================
# 5. CLEAN NON-FINITE NUMERIC VALUES
# ============================================================

for frame in [llm_review_df, real_validation_df, synthetic_validation_df]:
    numeric_cols = frame.select_dtypes(include=[np.number]).columns
    if len(numeric_cols) > 0:
        frame[numeric_cols] = frame[numeric_cols].replace([np.inf, -np.inf], np.nan)

# ============================================================
# 6. SAVE FILES
# ============================================================

production_file = "artha_sentinel_anomalies.csv"
real_validation_file = "artha_sentinel_anomalies_validation.csv"
synthetic_validation_file = "artha_sentinel_synthetic_validation.csv"

llm_review_df.to_csv(production_file, index=False)
real_validation_df.to_csv(real_validation_file, index=False)
synthetic_validation_df.to_csv(synthetic_validation_file, index=False)

# ============================================================
# 7. SUMMARY
# ============================================================

print("\nPRODUCTION ANOMALY CSV")
print("-" * 70)
print("Real dataset records:", len(final_df))
print("Review-worthy real records:", len(llm_review_df))
print("Review percentage:", f"{len(llm_review_df) / len(final_df) * 100:.2f}%")
print("LLM-facing columns:", len(llm_review_df.columns))
print("File:", production_file)

print("\nREAL VALIDATION CSV")
print("-" * 70)
print("Rows:", len(real_validation_df))
print("Columns:", len(real_validation_df.columns))
print("File:", real_validation_file)

print("\nSYNTHETIC VALIDATION CSV")
print("-" * 70)
print("Rows:", len(synthetic_validation_df))
print("Columns:", len(synthetic_validation_df.columns))
print("File:", synthetic_validation_file)

print("\nProduction risk distribution:")
print(llm_review_df["risk_band"].value_counts())

print("\n✅ STEP 9 COMPLETE")


STEP 9 — FINAL CSV EXPORT

PRODUCTION ANOMALY CSV
----------------------------------------------------------------------
Real dataset records: 23105
Review-worthy real records: 5741
Review percentage: 24.85%
LLM-facing columns: 48
File: artha_sentinel_anomalies.csv

REAL VALIDATION CSV
----------------------------------------------------------------------
Rows: 23105
Columns: 102
File: artha_sentinel_anomalies_validation.csv

SYNTHETIC VALIDATION CSV
----------------------------------------------------------------------
Rows: 300
Columns: 72
File: artha_sentinel_synthetic_validation.csv

Production risk distribution:
risk_band
LOW         3456
MODERATE    1898
HIGH         234
CRITICAL     153
Name: count, dtype: int64

✅ STEP 9 COMPLETE



PRODUCTION ANOMALY CSV
----------------------------------------------------------------------
Real dataset records: 23105
Review-worthy real records: 5741
Review percentage: 24.85%
LLM-facing columns: 48
File: artha_sentinel_anomalies.csv

REAL VALIDATION CSV
----------------------------------------------------------------------
Rows: 23105
Columns: 102
File: artha_sentinel_anomalies_validation.csv

SYNTHETIC VALIDATION CSV
----------------------------------------------------------------------
Rows: 300
Columns: 72
File: artha_sentinel_synthetic_validation.csv

Production risk distribution:
risk_band
LOW         3455
MODERATE    1899
HIGH         234
CRITICAL     153
Name: count, dtype: int64

✅ STEP 9 COMPLETE


In [42]:
final_csv = pd.read_csv("artha_sentinel_anomalies.csv")

In [43]:
fin_.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5741 entries, 0 to 5740
Data columns (total 48 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   work_id                        5741 non-null   int64  
 1   house                          5741 non-null   object 
 2   work_description               5692 non-null   object 
 3   primary_work_type              5741 non-null   object 
 4   secondary_work_type            177 non-null    object 
 5   work_subtype                   5741 non-null   object 
 6   category                       5739 non-null   object 
 7   mp_name                        5741 non-null   object 
 8   constituency                   5741 non-null   object 
 9   state                          5741 non-null   object 
 10  geo_district                   5741 non-null   object 
 11  location                       5741 non-null   object 
 12  cost                           5741 non-null   f

In [53]:
fin["risk_band"].value_counts()

risk_band
LOW         3456
MODERATE    1898
HIGH         234
CRITICAL     153
Name: count, dtype: int64